# Full Temporal Development Pipeline: NORMAL vs Synthetic LAG

This notebook documents the **full development pipeline for the isolated temporal branch** used during the thesis.

Its purpose is broader than the final isolated temporal experiment reported in the thesis. Here, the complete temporal pipeline is constructed from the processed Seamless Interaction dataset, including:

- discovery of clean dyadic conversations;
- deterministic eligibility filtering;
- a frozen 50-conversation temporal reference split;
- a disjoint 100-conversation inference split;
- synthetic temporal perturbations at `+1 s`, `+2 s`, and `+3 s`;
- VAD-turn merging;
- independent backchannel-like turn filtering;
- cleaned overlap features;
- signed local response-offset features;
- global Participant-B correction-shift features;
- frozen profile statistics;
- text-only Qwen2.5-Omni temporal reasoning.

The notebook therefore represents the **development stage** in the temporal branch:

```text
Full temporal development pipeline
        ↓
Feature inspection and selection
        ↓
Reduced selected temporal evidence
        ↓
Reported isolated temporal result
```

The richer representation explored here still exposes filtered turns and overlap-related information. In the subsequent reduced isolated-temporal notebook, these inputs are removed and the selected representation retains the local signed-offset statistics together with the global alignment-shift features. That reduced experiment is the one reported in the thesis.

No code in this notebook has been modified from the original development experiment; only the explanatory Markdown has been reorganized for clarity.

## Development Setup and Experimental Role

### Frozen reference set

The first deterministic set of **50 eligible conversations** is used only to estimate frozen temporal reference profiles:

- 50 NORMAL
- 50 LAG `+1 s`
- 50 LAG `+2 s`
- 50 LAG `+3 s`

The three synthetic LAG profiles remain separate rather than being pooled.

### Held-out development inference set

A different deterministic set of **100 conversations** is used only for inference:

- 100 NORMAL
- 30 LAG `+1 s`
- 40 LAG `+2 s`
- 40 LAG `+3 s`

The reference and inference conversation IDs are explicitly checked to be disjoint.

### Temporal representation explored in this development notebook

The development pipeline includes:

- the first 120 seconds of each interaction;
- VAD merging when gaps are at most `0.75 s`;
- independent removal of operational backchannel-like turns when:
  - turn duration is at most `1.0 s`, and
  - at least `80%` of that turn overlaps the other participant;
- cleaned overlap features;
- signed strict `A_end → B_start` offsets;
- deterministic summary statistics over those offsets;
- a global Participant-B correction search from `-6 s` to `+6 s` in `0.1 s` steps;
- text-only Qwen2.5-Omni classification.

The output remains binary: `NORMAL` or `LAG`.

This notebook is intentionally retained as the **full development record**. The later selected representation is more compact and excludes explicit filtered-turn and overlap inputs.

## 1. Install Dependencies

Install the libraries required for Qwen2.5-Omni text-only inference and the supporting temporal-analysis utilities.

In [ ]:
# Run this installation cell ONCE in a fresh Colab runtime.

!pip uninstall -y transformers
!pip install -q "transformers==4.57.6" accelerate bitsandbytes sentencepiece modelscope
!pip install -q qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

print("Installation complete. Select Runtime -> Restart session, then continue from the next cell.")


Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 141.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━

## 2. Mount Google Drive and Configure the Development Experiment

Mount persistent storage and define the paths and constants used throughout the temporal development pipeline.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from functools import lru_cache
import hashlib
import json
import random
import re

import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path(
    "/content/drive/MyDrive/seamless_download/data"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
NUM_REFERENCE_CONVERSATIONS = 50
NUM_INFERENCE_CONVERSATIONS = 100

INFERENCE_LAG_COUNTS = {
    1.0: 30,
    2.0: 35,
    3.0: 35,
}

REFERENCE_LAG_SECONDS = [
    1.0,
    2.0,
    3.0,
]

MAX_SECONDS = 120.0
MIN_TURNS_PER_PARTICIPANT = 5
MIN_SPEECH_SECONDS_PER_PARTICIPANT = 10.0

MERGE_GAP_SECONDS = 0.75
BOUNDARY_SAFE_MODE = False

PROMPT_VERSION = "mixed_lag_frozen_reference_v1"
MODEL_ID = "Qwen/Qwen2.5-Omni-7B"
MAX_NEW_TOKENS = 260

# None runs all 210 held-out inference cases.
PILOT_MAX_CASES = None

REFERENCE_SELECTION_PATH = OUT_DIR / "reference_50_conversations.json"
INFERENCE_SELECTION_PATH = OUT_DIR / "heldout_100_conversations.json"

REFERENCE_CASES_PATH = OUT_DIR / "reference_cases_normal_lag1_lag2_lag3.json"
INFERENCE_CASES_PATH = OUT_DIR / "inference_cases_100_normal_110_mixed_lag.json"

REFERENCE_FEATURE_CASES_PATH = OUT_DIR / "reference_feature_cases.json"
INFERENCE_FEATURE_CASES_PATH = OUT_DIR / "inference_feature_cases.json"

REFERENCE_BASE_STATS_PATH = OUT_DIR / "frozen_reference_base_statistics.json"
REFERENCE_SHIFT_STATS_PATH = OUT_DIR / "frozen_reference_global_shift_statistics.json"

GLOBAL_SHIFT_CACHE_PATH = OUT_DIR / "global_shift_feature_cache.json"

RESULTS_PATH = OUT_DIR / "qwen_mixed_lag_binary_results.json"
RESULTS_CSV_PATH = OUT_DIR / "qwen_mixed_lag_binary_results.csv"

assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}"
assert sum(INFERENCE_LAG_COUNTS.values()) == 100
assert NUM_REFERENCE_CONVERSATIONS + NUM_INFERENCE_CONVERSATIONS == 150

print("DATA_ROOT:", DATA_ROOT)
print("OUT_DIR:", OUT_DIR)
print("Reference conversations:", NUM_REFERENCE_CONVERSATIONS)
print("Held-out inference conversations:", NUM_INFERENCE_CONVERSATIONS)
print("Held-out lag counts:", INFERENCE_LAG_COUNTS)
print("Boundary-safe mode:", BOUNDARY_SAFE_MODE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_ROOT: /content/drive/MyDrive/seamless_download/data
OUT_DIR: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec
Reference conversations: 50
Held-out inference conversations: 100
Held-out lag counts: {1.0: 30, 2.0: 35, 3.0: 35}
Boundary-safe mode: False


## 3. Discover Structurally Clean Dyadic Conversations

Scan the processed Seamless Interaction dataset and identify ordinary two-participant conversations with participant-specific metadata.

The dedicated silent-participant pool is not part of this temporal experiment.

In [ ]:
def find_metadata_json(conversation_id: str, participant_id: str):
    participant_dir = DATA_ROOT / conversation_id / participant_id

    expected = participant_dir / f"{conversation_id}_{participant_id}.json"
    if expected.exists():
        return expected

    # Fallback for small filename differences.
    candidates = [
        p for p in sorted(participant_dir.glob("*.json"))
        if not p.name.endswith(".metadata.json")
    ]
    return candidates[0] if candidates else None


@lru_cache(maxsize=None)
def load_metadata_json(metadata_path_str: str):
    metadata_path = Path(metadata_path_str)
    with metadata_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def scan_normal_conversations():
    records = []

    for conv_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
        if conv_dir.name == "silent":
            continue

        participant_dirs = sorted(
            p for p in conv_dir.iterdir() if p.is_dir()
        )

        # Keep clean dyadic conversations only.
        if len(participant_dirs) != 2:
            continue

        participants = []

        for part_dir in participant_dirs:
            metadata_path = find_metadata_json(
                conv_dir.name,
                part_dir.name,
            )

            if metadata_path is None:
                participants = []
                break

            try:
                metadata = load_metadata_json(str(metadata_path))
            except Exception as exc:
                print("Could not read:", metadata_path, exc)
                participants = []
                break

            vad = metadata.get("metadata:vad", []) or []

            participants.append({
                "conversation_id": conv_dir.name,
                "participant_id": part_dir.name,
                "metadata_path": str(metadata_path),
                "num_raw_vad_entries": len(vad),
            })

        if len(participants) == 2:
            records.append({
                "conversation_id": conv_dir.name,
                "participants": participants,
            })

    return records


normal_conversations = scan_normal_conversations()

print("Clean dyadic normal conversations found:", len(normal_conversations))
print(json.dumps(normal_conversations[:2], indent=2))


Clean dyadic normal conversations found: 206
[
  {
    "conversation_id": "V00_S2017_I00001160",
    "participants": [
      {
        "conversation_id": "V00_S2017_I00001160",
        "participant_id": "P1273A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P1273A/V00_S2017_I00001160_P1273A.json",
        "num_raw_vad_entries": 49
      },
      {
        "conversation_id": "V00_S2017_I00001160",
        "participant_id": "P2072A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001160/P2072A/V00_S2017_I00001160_P2072A.json",
        "num_raw_vad_entries": 33
      }
    ]
  },
  {
    "conversation_id": "V00_S2017_I00001161",
    "participants": [
      {
        "conversation_id": "V00_S2017_I00001161",
        "participant_id": "P1273A",
        "metadata_path": "/content/drive/MyDrive/seamless_download/data/V00_S2017_I00001161/P1273A/V00_S2017_I00001161_P1273A.json",
        "num_raw_vad_entries": 3

## 4. Extract, Clip, and Merge VAD Speaking Ranges

Participant VAD intervals are restricted to the first 120 seconds and consecutive regions separated by at most `0.75 s` are merged.

These merged speaking ranges form the raw temporal representation used by the remainder of the pipeline.

In [ ]:
def merge_ranges(ranges, max_gap=MERGE_GAP_SECONDS):
    if not ranges:
        return []

    clean = sorted(
        [
            {
                "start": float(r["start"]),
                "end": float(r["end"]),
            }
            for r in ranges
            if float(r["end"]) > float(r["start"])
        ],
        key=lambda r: (r["start"], r["end"]),
    )

    if not clean:
        return []

    merged = [dict(clean[0])]

    for current in clean[1:]:
        previous = merged[-1]
        gap = current["start"] - previous["end"]

        if gap <= float(max_gap):
            previous["end"] = max(
                previous["end"],
                current["end"],
            )
        else:
            merged.append(dict(current))

    return [
        {
            "start": round(r["start"], 2),
            "end": round(r["end"], 2),
        }
        for r in merged
    ]


def get_full_merged_vad_turns(
    metadata_path,
    max_seconds=MAX_SECONDS,
):
    metadata = load_metadata_json(str(metadata_path))
    raw_vad = metadata.get("metadata:vad", []) or []

    clipped = []

    for item in raw_vad:
        try:
            start = float(item.get("start", 0.0))
            end = float(item.get("end", start))
        except (TypeError, ValueError):
            continue

        clipped_start = max(0.0, start)
        clipped_end = min(float(max_seconds), end)

        if clipped_end > clipped_start:
            clipped.append({
                "start": clipped_start,
                "end": clipped_end,
            })

    return merge_ranges(
        clipped,
        max_gap=MERGE_GAP_SECONDS,
    )


def shift_ranges(ranges, shift_seconds):
    return [
        {
            "start": round(float(r["start"]) + shift_seconds, 2),
            "end": round(float(r["end"]) + shift_seconds, 2),
        }
        for r in ranges
    ]


def clip_and_localize_ranges(
    ranges,
    global_start,
    global_end,
):
    local = []

    for r in ranges:
        start = max(float(r["start"]), float(global_start))
        end = min(float(r["end"]), float(global_end))

        if end <= start:
            continue

        local.append({
            "start": round(start - global_start, 2),
            "end": round(end - global_start, 2),
        })

    return local


def total_speech_seconds(ranges):
    return round(
        sum(float(r["end"]) - float(r["start"]) for r in ranges),
        2,
    )


## 5. Apply Eligibility Criteria and Construct Disjoint Reference / Inference Splits

A conversation is retained only when both participants satisfy the temporal eligibility requirements implemented below:

- each participant has at least the required minimum number of merged turns;
- each participant contributes at least the required minimum total speech duration.

The resulting eligible pool contains **167 conversations**.

From this pool:

- **50 conversations** are deterministically selected for frozen temporal-reference estimation;
- **100 different conversations** are selected for development inference.

The notebook verifies that the two source-conversation sets are disjoint.

In [ ]:
eligible_conversations = []

for conv in normal_conversations:
    participants = conv["participants"]

    turns_0 = get_full_merged_vad_turns(
        participants[0]["metadata_path"]
    )

    turns_1 = get_full_merged_vad_turns(
        participants[1]["metadata_path"]
    )

    if not turns_0 or not turns_1:
        continue

    if (
        len(turns_0) < MIN_TURNS_PER_PARTICIPANT
        or len(turns_1) < MIN_TURNS_PER_PARTICIPANT
        or total_speech_seconds(turns_0) < MIN_SPEECH_SECONDS_PER_PARTICIPANT
        or total_speech_seconds(turns_1) < MIN_SPEECH_SECONDS_PER_PARTICIPANT
    ):
        continue

    record = dict(conv)
    record["participant_0_full_turns"] = turns_0
    record["participant_1_full_turns"] = turns_1
    eligible_conversations.append(record)

required_conversations = (
    NUM_REFERENCE_CONVERSATIONS
    + NUM_INFERENCE_CONVERSATIONS
)

assert len(eligible_conversations) >= required_conversations, (
    f"Only {len(eligible_conversations)} eligible conversations were found; "
    f"{required_conversations} are required."
)

# Exact recreation of the original deterministic 50-conversation sample.
reference_rng = random.Random(RANDOM_SEED)

selected_reference_conversations = reference_rng.sample(
    eligible_conversations,
    NUM_REFERENCE_CONVERSATIONS,
)


def orient_conversation(conv, reverse_direction):
    if reverse_direction:
        participant_A = conv["participants"][1]
        participant_B = conv["participants"][0]
        turns_A_full = conv["participant_1_full_turns"]
        turns_B_full = conv["participant_0_full_turns"]
    else:
        participant_A = conv["participants"][0]
        participant_B = conv["participants"][1]
        turns_A_full = conv["participant_0_full_turns"]
        turns_B_full = conv["participant_1_full_turns"]

    return {
        "conversation_id": conv["conversation_id"],
        "participant_A": participant_A,
        "participant_B": participant_B,
        "turns_A_full_0_120": turns_A_full,
        "turns_B_full_0_120": turns_B_full,
    }


reference_selection_manifest = []

for conv in selected_reference_conversations:
    reverse_direction = bool(reference_rng.getrandbits(1))

    reference_selection_manifest.append(
        orient_conversation(
            conv,
            reverse_direction,
        )
    )

reference_ids = {
    record["conversation_id"]
    for record in reference_selection_manifest
}

remaining_conversations = [
    conv
    for conv in eligible_conversations
    if conv["conversation_id"] not in reference_ids
]

inference_rng = random.Random(RANDOM_SEED + 2000)

selected_inference_conversations = inference_rng.sample(
    remaining_conversations,
    NUM_INFERENCE_CONVERSATIONS,
)

inference_selection_manifest = []

for conv in selected_inference_conversations:
    reverse_direction = bool(inference_rng.getrandbits(1))

    inference_selection_manifest.append(
        orient_conversation(
            conv,
            reverse_direction,
        )
    )

inference_ids = {
    record["conversation_id"]
    for record in inference_selection_manifest
}

assert reference_ids.isdisjoint(inference_ids)
assert len(reference_selection_manifest) == 50
assert len(inference_selection_manifest) == 100

# Assign 110 lag cases with only the unavoidable 10 repeated source conversations.
assignment_rng = random.Random(RANDOM_SEED + 3000)
assignment_order = list(range(NUM_INFERENCE_CONVERSATIONS))
assignment_rng.shuffle(assignment_order)

lag_3_indices = set(assignment_order[:40])
lag_2_indices = set(assignment_order[40:80])
lag_1_indices = set(assignment_order[80:100])
lag_1_indices.update(
    assignment_rng.sample(
        assignment_order[:80],
        10,
    )
)

for index, record in enumerate(inference_selection_manifest):
    assigned_lags = []

    if index in lag_1_indices:
        assigned_lags.append(1.0)

    if index in lag_2_indices:
        assigned_lags.append(2.0)

    if index in lag_3_indices:
        assigned_lags.append(3.0)

    record["assigned_lag_seconds"] = assigned_lags

assert sum(1.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 30
assert sum(2.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 40
assert sum(3.0 in r["assigned_lag_seconds"] for r in inference_selection_manifest) == 40

REFERENCE_SELECTION_PATH.write_text(
    json.dumps(
        reference_selection_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_SELECTION_PATH.write_text(
    json.dumps(
        inference_selection_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# Optional strict verification against the previous +1 s selection manifest.
legacy_selection_path = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_lag_1sec/"
    "selected_50_conversations.json"
)

if legacy_selection_path.exists():
    legacy_manifest = json.loads(
        legacy_selection_path.read_text(
            encoding="utf-8"
        )
    )

    legacy_signature = [
        (
            record["conversation_id"],
            record["participant_A"]["participant_id"],
            record["participant_B"]["participant_id"],
        )
        for record in legacy_manifest
    ]

    current_signature = [
        (
            record["conversation_id"],
            record["participant_A"]["participant_id"],
            record["participant_B"]["participant_id"],
        )
        for record in reference_selection_manifest
    ]

    assert legacy_signature == current_signature, (
        "The recreated reference split does not match the previous 50-conversation split."
    )

    print("Verified: reference 50 exactly match the previous +1 s selection.")
else:
    print(
        "Legacy selection file not found. "
        "The original deterministic selection algorithm was reproduced exactly."
    )

print("Eligible conversations:", len(eligible_conversations))
print("Reference conversations:", len(reference_selection_manifest))
print("Held-out conversations:", len(inference_selection_manifest))
print("Reference/inference overlap:", len(reference_ids & inference_ids))
print("Saved reference selection:", REFERENCE_SELECTION_PATH)
print("Saved inference selection:", INFERENCE_SELECTION_PATH)

Verified: reference 50 exactly match the previous +1 s selection.
Eligible conversations: 167
Reference conversations: 50
Held-out conversations: 100
Reference/inference overlap: 0
Saved reference selection: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reference_50_conversations.json
Saved inference selection: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/heldout_100_conversations.json


## 6. Construct NORMAL and Synthetic LAG Cases

Each selected conversation is converted into observed temporal cases.

For the frozen reference set, every source conversation produces separate profiles for:

- NORMAL
- LAG `+1 s`
- LAG `+2 s`
- LAG `+3 s`

For the held-out development set, the original NORMAL timeline is retained and Participant-B speaking ranges are synthetically shifted according to the assigned delay profile.

Only the VAD-derived temporal timeline is shifted; this stage operates on temporal evidence rather than modifying the underlying audiovisual stream.

In [ ]:
def stable_case_id(
    conversation_id,
    split_name,
    variant,
    lag_seconds,
):
    payload = (
        f"{conversation_id}|{split_name}|{variant}|"
        f"{float(lag_seconds):.1f}|{MAX_SECONDS}|"
        f"{MERGE_GAP_SECONDS}|{BOUNDARY_SAFE_MODE}|"
        f"{PROMPT_VERSION}"
    )

    digest = hashlib.sha1(
        payload.encode("utf-8")
    ).hexdigest()[:12]

    return f"case_{digest}"


def build_observed_case(
    record,
    split_name,
    lag_seconds,
):
    lag_seconds = float(lag_seconds)

    full_A = record["turns_A_full_0_120"]
    full_B = record["turns_B_full_0_120"]

    observed_B = (
        shift_ranges(full_B, lag_seconds)
        if lag_seconds > 0
        else full_B
    )

    if BOUNDARY_SAFE_MODE:
        boundary_margin = max(REFERENCE_LAG_SECONDS)
        analysis_start = float(boundary_margin)
        analysis_end = float(MAX_SECONDS - boundary_margin)
    else:
        analysis_start = 0.0
        analysis_end = float(MAX_SECONDS)

    analysis_duration = analysis_end - analysis_start

    turns_A = clip_and_localize_ranges(
        full_A,
        analysis_start,
        analysis_end,
    )

    turns_B = clip_and_localize_ranges(
        observed_B,
        analysis_start,
        analysis_end,
    )

    is_lag = lag_seconds > 0

    profile = (
        f"LAG_{int(lag_seconds)}"
        if is_lag
        else "NORMAL"
    )

    variant = (
        f"lag_{int(lag_seconds)}sec"
        if is_lag
        else "normal"
    )

    return {
        "case_id": stable_case_id(
            record["conversation_id"],
            split_name,
            variant,
            lag_seconds,
        ),
        "split_name": split_name,
        "conversation_id": record["conversation_id"],
        "participant_A_id": record["participant_A"]["participant_id"],
        "participant_B_id": record["participant_B"]["participant_id"],
        "variant": variant,
        "reference_profile": profile,
        "gold_label": "LAG" if is_lag else "NORMAL",
        "lag_seconds": lag_seconds,
        "analysis_global_start": analysis_start,
        "analysis_global_end": analysis_end,
        "analysis_duration": analysis_duration,
        "merge_gap_seconds": MERGE_GAP_SECONDS,
        "boundary_safe_mode": BOUNDARY_SAFE_MODE,
        "turns_A": turns_A,
        "turns_B": turns_B,
    }


reference_cases = []

for record in reference_selection_manifest:
    reference_cases.append(
        build_observed_case(
            record,
            split_name="reference",
            lag_seconds=0.0,
        )
    )

    for lag_seconds in REFERENCE_LAG_SECONDS:
        reference_cases.append(
            build_observed_case(
                record,
                split_name="reference",
                lag_seconds=lag_seconds,
            )
        )

inference_cases = []

for record in inference_selection_manifest:
    inference_cases.append(
        build_observed_case(
            record,
            split_name="inference",
            lag_seconds=0.0,
        )
    )

    for lag_seconds in record["assigned_lag_seconds"]:
        inference_cases.append(
            build_observed_case(
                record,
                split_name="inference",
                lag_seconds=lag_seconds,
            )
        )

inference_case_rng = random.Random(RANDOM_SEED + 4000)
inference_case_rng.shuffle(inference_cases)

assert len(reference_cases) == 200
assert len(inference_cases) == 210

assert sum(c["reference_profile"] == "NORMAL" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_1" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_2" for c in reference_cases) == 50
assert sum(c["reference_profile"] == "LAG_3" for c in reference_cases) == 50

assert sum(c["reference_profile"] == "NORMAL" for c in inference_cases) == 100
assert sum(c["reference_profile"] == "LAG_1" for c in inference_cases) == 30
assert sum(c["reference_profile"] == "LAG_2" for c in inference_cases) == 40
assert sum(c["reference_profile"] == "LAG_3" for c in inference_cases) == 40

reference_case_conversation_ids = {
    case["conversation_id"]
    for case in reference_cases
}

inference_case_conversation_ids = {
    case["conversation_id"]
    for case in inference_cases
}

assert reference_case_conversation_ids.isdisjoint(
    inference_case_conversation_ids
)

REFERENCE_CASES_PATH.write_text(
    json.dumps(
        reference_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_CASES_PATH.write_text(
    json.dumps(
        inference_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Reference cases:", len(reference_cases))
print(pd.Series([c["reference_profile"] for c in reference_cases]).value_counts().sort_index())

print("\nHeld-out inference cases:", len(inference_cases))
print(pd.Series([c["reference_profile"] for c in inference_cases]).value_counts().sort_index())

print(
    "\nReference/inference conversation overlap:",
    len(reference_case_conversation_ids & inference_case_conversation_ids),
)

Reference cases: 200
LAG_1     50
LAG_2     50
LAG_3     50
NORMAL    50
Name: count, dtype: int64

Held-out inference cases: 210
LAG_1      30
LAG_2      40
LAG_3      40
NORMAL    100
Name: count, dtype: int64

Reference/inference conversation overlap: 0


## 7. Load Qwen2.5-Omni Thinker

Load the same Qwen2.5-Omni Thinker used for the temporal reasoning experiments.

At this stage Qwen operates over structured textual temporal evidence rather than raw audiovisual input.

In [ ]:
import torch
from transformers import (
    Qwen2_5OmniThinkerForConditionalGeneration,
    Qwen2_5OmniProcessor,
)

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU runtime before loading Qwen2.5-Omni-7B."
    )

print("GPU:", torch.cuda.get_device_name(0))
assert MODEL_ID == "Qwen/Qwen2.5-Omni-7B"

# Load the already downloaded model directly from Google Drive.
MODEL_PATH = "/content/drive/MyDrive/Qwen2.5-Omni-7B"
print("Loading model from:", MODEL_PATH)

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True,
)

# processor = Qwen2_5OmniProcessor.from_pretrained(
#     MODEL_PATH,
#     local_files_only=True,
# )

model.eval()
print("Loaded:", MODEL_ID)


`torch_dtype` is deprecated! Use `dtype` instead!


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Loading model from: /content/drive/MyDrive/Qwen2.5-Omni-7B


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-Omni-7B


In [ ]:
processor = Qwen2_5OmniProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    use_fast=False,
)

## 8. JSON Parsing and Deterministic Text-Only Inference

Define the structured-output parser and deterministic Qwen text-only inference helper used by the temporal reasoner.

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")

    if start == -1:
        return {
            "parse_error": True,
            "raw_output": text,
        }

    depth = 0

    for index in range(start, len(text)):
        if text[index] == "{":
            depth += 1
        elif text[index] == "}":
            depth -= 1

            if depth == 0:
                candidate = text[start:index + 1]

                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {
        "parse_error": True,
        "raw_output": text,
    }


# def qwen_text_only(prompt: str, max_new_tokens=MAX_NEW_TOKENS):
#     messages = [
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": prompt,
#                 }
#             ],
#         }
#     ]

#     inputs = processor.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         tokenize=True,
#         return_dict=True,
#         return_tensors="pt",
#         processor_kwargs={
#             "padding": True,
#         },
#     )

#     inputs = {
#         key: value.to(model.device)
#         if hasattr(value, "to")
#         else value
#         for key, value in inputs.items()
#     }

#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#         )

#     generated = processor.batch_decode(
#         output_ids[
#             :,
#             inputs["input_ids"].shape[1]:,
#         ],
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False,
#     )[0]

#     return generated

def qwen_text_only(prompt: str, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        if hasattr(value, "to")
        else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[
            :,
            inputs["input_ids"].shape[1]:
        ],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


## 9. Signed Local Response-Offset Utilities

Define the shared utilities for extracting signed strict transition offsets around participant turn boundaries.

Positive offsets represent delayed Participant-B starts after Participant A ends; negative offsets represent short pre-end starts where Participant B begins shortly before Participant A finishes.

In [ ]:
import numpy as np
import pandas as pd
from bisect import bisect_left

# Exact limits used by the final experiment.
DURATION_ONLY_MAX_POST_DELAY_SECONDS = 6.0
DURATION_ONLY_MAX_PRESTART_SECONDS = 2.0

def duration_only_sorted_turns(turns):
    """
    Return sorted copies of the supplied turns.
    """

    return sorted(
        [
            {
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            }
            for turn in turns
            if (
                float(turn["end"])
                > float(turn["start"])
            )
        ],
        key=lambda turn: (
            turn["start"],
            turn["end"],
        ),
    )


def duration_only_single_overlap_seconds(
    turn_A,
    turn_B,
):
    return max(
        0.0,
        min(
            float(turn_A["end"]),
            float(turn_B["end"]),
        )
        - max(
            float(turn_A["start"]),
            float(turn_B["start"]),
        ),
    )


def duration_only_total_overlap_seconds(
    turns_A,
    turns_B,
):
    total = 0.0

    for A_turn in turns_A:
        for B_turn in turns_B:
            total += (
                duration_only_single_overlap_seconds(
                    A_turn,
                    B_turn,
                )
            )

    return total


def duration_only_signed_strict_events(
    turns_A,
    turns_B,
    max_post_delay_seconds=(
        DURATION_ONLY_MAX_POST_DELAY_SECONDS
    ),
    max_prestart_seconds=(
        DURATION_ONLY_MAX_PRESTART_SECONDS
    ),
):
    """
    Create at most one signed event for each relevant A_end.

    Negative event:

        A_start < B_start < A_end < B_end

        B starts up to max_prestart_seconds before A_end.

    Positive event:

        A_end <= B_start

        B starts up to max_post_delay_seconds after A_end,
        and A does not restart before or at B_start.

    signed_offset = B_start - A_end
    """

    A_turns = duration_only_sorted_turns(
        turns_A
    )

    B_turns = duration_only_sorted_turns(
        turns_B
    )

    B_starts = [
        float(turn["start"])
        for turn in B_turns
    ]

    events = []

    for A_index, A_turn in enumerate(A_turns):
        A_start = float(
            A_turn["start"]
        )

        A_end = float(
            A_turn["end"]
        )

        # ----------------------------------------------------
        # Negative event:
        # B is already active when A ends.
        # ----------------------------------------------------

        active_B_candidates = []

        for B_index, B_turn in enumerate(B_turns):
            B_start = float(
                B_turn["start"]
            )

            B_end = float(
                B_turn["end"]
            )

            if B_start < A_end < B_end:
                active_B_candidates.append(
                    (
                        B_index,
                        B_start,
                        B_end,
                    )
                )

        if active_B_candidates:
            recent_candidates = [
                (
                    B_index,
                    B_start,
                    B_end,
                )
                for (
                    B_index,
                    B_start,
                    B_end,
                ) in active_B_candidates
                if (
                    A_start < B_start < A_end
                    and (
                        A_end - B_start
                        <= float(
                            max_prestart_seconds
                        )
                    )
                )
            ]

            if recent_candidates:
                # Keep the B start closest to A_end.
                (
                    B_index,
                    B_start,
                    B_end,
                ) = max(
                    recent_candidates,
                    key=lambda item: item[1],
                )

                signed_offset = (
                    B_start - A_end
                )

                events.append({
                    "event_type": (
                        "B_STARTS_SHORTLY_BEFORE_A_END"
                    ),

                    "A_turn_index": int(
                        A_index
                    ),

                    "A_start": round(
                        A_start,
                        2,
                    ),

                    "A_end": round(
                        A_end,
                        2,
                    ),

                    "B_turn_index": int(
                        B_index
                    ),

                    "B_start": round(
                        B_start,
                        2,
                    ),

                    "B_end": round(
                        B_end,
                        2,
                    ),

                    "signed_offset_seconds": round(
                        signed_offset,
                        2,
                    ),
                })

            # Exact same priority as previous experiment:
            # when B is already active at A_end, do not also
            # search for a future positive event.
            continue

        # ----------------------------------------------------
        # Positive event:
        # B begins after A ends.
        # ----------------------------------------------------

        B_index = bisect_left(
            B_starts,
            A_end,
        )

        if B_index >= len(B_starts):
            continue

        B_start = float(
            B_starts[B_index]
        )

        signed_offset = (
            B_start - A_end
        )

        if signed_offset < 0:
            continue

        if (
            signed_offset
            > float(max_post_delay_seconds)
        ):
            continue

        next_A_start = (
            float(
                A_turns[
                    A_index + 1
                ]["start"]
            )
            if (
                A_index + 1
                < len(A_turns)
            )
            else None
        )

        # Same strict exclusion as previous experiment.
        if (
            next_A_start is not None
            and next_A_start <= B_start
        ):
            continue

        B_end = float(
            B_turns[B_index]["end"]
        )

        events.append({
            "event_type": (
                "B_STARTS_AFTER_A_END"
            ),

            "A_turn_index": int(
                A_index
            ),

            "A_start": round(
                A_start,
                2,
            ),

            "A_end": round(
                A_end,
                2,
            ),

            "B_turn_index": int(
                B_index
            ),

            "B_start": round(
                B_start,
                2,
            ),

            "B_end": round(
                B_end,
                2,
            ),

            "signed_offset_seconds": round(
                signed_offset,
                2,
            ),
        })

    return events


def compact_turns_text(turns):
    compact = [
        [
            round(float(turn["start"]), 2),
            round(float(turn["end"]), 2),
        ]
        for turn in turns
    ]

    return json.dumps(
        compact,
        ensure_ascii=False,
        separators=(",", ":"),
    )

## 10. Independent Backchannel-Like Turn Filtering

Short overlapping turns can behave like acknowledgements or backchannels rather than primary turn transitions.

This development pipeline therefore applies an operational filter independently to each observed case. A turn is removed only when:

- its duration is at most `1.0 s`; and
- at least `80%` of its duration overlaps the other participant.

The filter uses only the observed speaking ranges. It does not use the NORMAL/LAG label.

In [ ]:
# ============================================================
# NEW EXPERIMENT
#
# Same pipeline as the duration-only signed-offset experiment.
#
# ONLY CHANGE:
#
# Independently remove operational backchannel-like turns:
#
#   duration <= 1.0 sec
#   AND
#   overlap fraction >= 80%
#
# Filtering is performed independently on every observed
# NORMAL or LAG case.
#
# No matching NORMAL case is used.
# No gold label is used during filtering/feature extraction.
# ============================================================

import json
import numpy as np
import pandas as pd


INDEPENDENT_BC_EXPERIMENT_VERSION = (
    "guided_independent_backchannel_filter_signed_offsets_v1"
)

INDEPENDENT_BC_MAX_DURATION_SECONDS = 1.0

INDEPENDENT_BC_MIN_OVERLAP_FRACTION = 0.80

INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS = 1.5

# Same signed-offset limits as the previous experiment
INDEPENDENT_BC_MAX_POST_DELAY_SECONDS = 6.0
INDEPENDENT_BC_MAX_PRESTART_SECONDS = 2.0

# None = all 100 cases
# Set to 4 for a quick pilot
INDEPENDENT_BC_PILOT_MAX_CASES = None

INDEPENDENT_BC_MAX_NEW_TOKENS = 260


INDEPENDENT_BC_CASES_PATH = (
    OUT_DIR
    / "guided_independent_backchannel_signed_offset_cases.json"
)

INDEPENDENT_BC_RESULTS_PATH = (
    OUT_DIR
    / "qwen_guided_independent_backchannel_signed_offset_results.json"
)

INDEPENDENT_BC_RESULTS_CSV_PATH = (
    OUT_DIR
    / "qwen_guided_independent_backchannel_signed_offset_results.csv"
)


INDEPENDENT_BC_PROMPT_VERSION = (
    f"{INDEPENDENT_BC_EXPERIMENT_VERSION}"
    f"_duration{INDEPENDENT_BC_MAX_DURATION_SECONDS}"
    f"_overlap{INDEPENDENT_BC_MIN_OVERLAP_FRACTION}"
    f"_threshold{INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS}"
    f"_postmax{INDEPENDENT_BC_MAX_POST_DELAY_SECONDS}"
    f"_premax{INDEPENDENT_BC_MAX_PRESTART_SECONDS}"
)


print(
    "Independent backchannel prompt version:",
    INDEPENDENT_BC_PROMPT_VERSION,
)


# ============================================================
# Compute the fraction of one target turn covered by
# the other participant's turns
# ============================================================

def independent_bc_overlap_fraction(
    target_turn,
    other_turns,
):
    """
    Return the fraction of target_turn covered by the union
    of the other participant's speaking turns.

    Using the union prevents accidental double counting.
    """

    target_start = float(
        target_turn["start"]
    )

    target_end = float(
        target_turn["end"]
    )

    target_duration = (
        target_end - target_start
    )

    if target_duration <= 0:
        return 0.0

    intersections = []

    for other_turn in other_turns:
        overlap_start = max(
            target_start,
            float(other_turn["start"]),
        )

        overlap_end = min(
            target_end,
            float(other_turn["end"]),
        )

        if overlap_end > overlap_start:
            intersections.append(
                (
                    overlap_start,
                    overlap_end,
                )
            )

    if not intersections:
        return 0.0

    intersections = sorted(
        intersections,
        key=lambda interval: (
            interval[0],
            interval[1],
        ),
    )

    merged_intersections = [
        list(intersections[0])
    ]

    for start, end in intersections[1:]:
        previous = merged_intersections[-1]

        if start <= previous[1]:
            previous[1] = max(
                previous[1],
                end,
            )
        else:
            merged_intersections.append(
                [start, end]
            )

    covered_seconds = sum(
        end - start
        for start, end
        in merged_intersections
    )

    return float(
        min(
            1.0,
            covered_seconds / target_duration,
        )
    )


# ============================================================
# Filter one participant independently
# ============================================================

def independent_bc_filter_stream(
    target_turns,
    other_turns,
    max_duration_seconds=(
        INDEPENDENT_BC_MAX_DURATION_SECONDS
    ),
    min_overlap_fraction=(
        INDEPENDENT_BC_MIN_OVERLAP_FRACTION
    ),
):
    """
    A target turn is removed only when:

        duration <= 1 sec
        AND
        overlap fraction >= 80%

    No semantic information or gold label is used.
    """

    target_turns = duration_only_sorted_turns(
        target_turns
    )

    other_turns = duration_only_sorted_turns(
        other_turns
    )

    filtered_turns = []
    removed_turns = []
    removed_indices = []

    for index, turn in enumerate(target_turns):
        duration = (
            float(turn["end"])
            - float(turn["start"])
        )

        overlap_fraction = (
            independent_bc_overlap_fraction(
                turn,
                other_turns,
            )
        )

        is_backchannel_like = (
            duration
            <= float(max_duration_seconds) + 1e-9
            and overlap_fraction
            >= float(min_overlap_fraction) - 1e-9
        )

        if is_backchannel_like:
            removed_indices.append(
                int(index)
            )

            removed_turns.append({
                "turn_index": int(index),

                "start": round(
                    float(turn["start"]),
                    3,
                ),

                "end": round(
                    float(turn["end"]),
                    3,
                ),

                "duration": round(
                    duration,
                    3,
                ),

                "overlap_fraction": round(
                    overlap_fraction,
                    4,
                ),

                "overlap_percent": round(
                    100.0 * overlap_fraction,
                    2,
                ),
            })

        else:
            filtered_turns.append({
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            })

    return {
        "original_turns": target_turns,
        "filtered_turns": filtered_turns,

        "removed_indices": removed_indices,
        "removed_turns": removed_turns,

        "num_original_turns": len(
            target_turns
        ),

        "num_filtered_turns": len(
            filtered_turns
        ),

        "num_removed_turns": len(
            removed_turns
        ),
    }


# ============================================================
# Filter each observed case independently
# ============================================================

def independently_filter_case_backchannels(
    case,
):
    """
    Detect A backchannels against the observed B turns.

    Detect B backchannels against the observed A turns.

    Both decisions are made from the original observed pair,
    before either participant is filtered.

    The function does not inspect:
        - gold_label
        - variant
        - lag_seconds
        - matching NORMAL cases
    """

    original_A = duration_only_sorted_turns(
        case["turns_A"]
    )

    original_B = duration_only_sorted_turns(
        case["turns_B"]
    )

    A_result = independent_bc_filter_stream(
        target_turns=original_A,
        other_turns=original_B,
    )

    B_result = independent_bc_filter_stream(
        target_turns=original_B,
        other_turns=original_A,
    )

    return {
        "original_turns_A": original_A,
        "original_turns_B": original_B,

        "filtered_turns_A": A_result[
            "filtered_turns"
        ],

        "filtered_turns_B": B_result[
            "filtered_turns"
        ],

        "num_original_A_turns": A_result[
            "num_original_turns"
        ],

        "num_original_B_turns": B_result[
            "num_original_turns"
        ],

        "num_filtered_A_turns": A_result[
            "num_filtered_turns"
        ],

        "num_filtered_B_turns": B_result[
            "num_filtered_turns"
        ],

        "num_removed_A_backchannels": A_result[
            "num_removed_turns"
        ],

        "num_removed_B_backchannels": B_result[
            "num_removed_turns"
        ],

        "removed_A_backchannel_indices": A_result[
            "removed_indices"
        ],

        "removed_B_backchannel_indices": B_result[
            "removed_indices"
        ],

        "removed_A_backchannels": A_result[
            "removed_turns"
        ],

        "removed_B_backchannels": B_result[
            "removed_turns"
        ],
    }

Independent backchannel prompt version: guided_independent_backchannel_filter_signed_offsets_v1_duration1.0_overlap0.8_threshold1.5_postmax6.0_premax2.0


## 11. Extract the Rich Development Temporal Feature Set

After independent backchannel filtering, the pipeline computes:

- filtered participant turns;
- cleaned speaking overlap;
- signed strict local response offsets;
- summary statistics of the signed-offset distribution.

This is the **richer development representation**. Some of these inputs—most importantly explicit filtered turns and overlap—are removed in the later reduced selected representation.

In [ ]:
# ============================================================
# Overlap after independent backchannel filtering
# ============================================================

def compute_independent_bc_overlap_features(
    original_turns_A,
    original_turns_B,
    filtered_turns_A,
    filtered_turns_B,
    timeline_duration,
):
    raw_overlap = (
        duration_only_total_overlap_seconds(
            original_turns_A,
            original_turns_B,
        )
    )

    clean_overlap = (
        duration_only_total_overlap_seconds(
            filtered_turns_A,
            filtered_turns_B,
        )
    )

    removed_overlap = max(
        0.0,
        raw_overlap - clean_overlap,
    )

    duration = float(
        timeline_duration
    )

    overlap_ratio = (
        clean_overlap / duration
        if duration > 0
        else 0.0
    )

    return {
        "raw_overlap_seconds": round(
            raw_overlap,
            2,
        ),

        "backchannel_overlap_removed_seconds": round(
            removed_overlap,
            2,
        ),

        "clean_overlap_seconds": round(
            clean_overlap,
            2,
        ),

        "clean_overlap_ratio": round(
            overlap_ratio,
            4,
        ),

        "clean_overlap_percent": round(
            100.0 * overlap_ratio,
            2,
        ),
    }


# ============================================================
# Summarize the exact same signed-offset distribution
# ============================================================

def summarize_independent_bc_signed_offsets(
    events,
):
    offsets = np.asarray(
        [
            event["signed_offset_seconds"]
            for event in events
        ],
        dtype=float,
    )

    num_negative = sum(
        event["event_type"]
        == "B_STARTS_SHORTLY_BEFORE_A_END"
        for event in events
    )

    num_positive = sum(
        event["event_type"]
        == "B_STARTS_AFTER_A_END"
        for event in events
    )

    if len(offsets) == 0:
        return {
            "signed_strict_offsets_seconds": [],
            "num_signed_strict_offsets": 0,

            "num_negative_preend_offsets": 0,
            "num_positive_postend_offsets": 0,

            "offset_mean_seconds": None,
            "offset_median_seconds": None,
            "offset_min_seconds": None,
            "offset_max_seconds": None,
            "offset_p75_seconds": None,
            "offset_p90_seconds": None,

            "num_offsets_above_1_5_seconds": 0,

            "percent_offsets_above_1_5_seconds": (
                None
            ),
        }

    num_above = int(
        np.sum(
            offsets
            > INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS
        )
    )

    return {
        "signed_strict_offsets_seconds": (
            offsets.round(2).tolist()
        ),

        "num_signed_strict_offsets": int(
            len(offsets)
        ),

        "num_negative_preend_offsets": int(
            num_negative
        ),

        "num_positive_postend_offsets": int(
            num_positive
        ),

        "offset_mean_seconds": round(
            float(np.mean(offsets)),
            2,
        ),

        "offset_median_seconds": round(
            float(np.median(offsets)),
            2,
        ),

        "offset_min_seconds": round(
            float(np.min(offsets)),
            2,
        ),

        "offset_max_seconds": round(
            float(np.max(offsets)),
            2,
        ),

        "offset_p75_seconds": round(
            float(np.percentile(offsets, 75)),
            2,
        ),

        "offset_p90_seconds": round(
            float(np.percentile(offsets, 90)),
            2,
        ),

        "num_offsets_above_1_5_seconds": (
            num_above
        ),

        "percent_offsets_above_1_5_seconds": round(
            100.0
            * num_above
            / len(offsets),
            1,
        ),
    }


# ============================================================
# Compute all features for one observed case
# ============================================================

def compute_independent_bc_features(
    case,
):
    filtered = (
        independently_filter_case_backchannels(
            case
        )
    )

    overlap = (
        compute_independent_bc_overlap_features(
            filtered["original_turns_A"],
            filtered["original_turns_B"],

            filtered["filtered_turns_A"],
            filtered["filtered_turns_B"],

            timeline_duration=case[
                "analysis_duration"
            ],
        )
    )

    # Exact same strict signed-offset logic
    # as the previous duration-only experiment
    events = duration_only_signed_strict_events(
        filtered["filtered_turns_A"],
        filtered["filtered_turns_B"],

        max_post_delay_seconds=(
            INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
        ),

        max_prestart_seconds=(
            INDEPENDENT_BC_MAX_PRESTART_SECONDS
        ),
    )

    distribution = (
        summarize_independent_bc_signed_offsets(
            events
        )
    )

    return {
        **overlap,
        **distribution,

        "backchannel_max_duration_seconds": (
            INDEPENDENT_BC_MAX_DURATION_SECONDS
        ),

        "backchannel_min_overlap_fraction": (
            INDEPENDENT_BC_MIN_OVERLAP_FRACTION
        ),

        "independent_filtering": True,

        "num_original_A_turns": filtered[
            "num_original_A_turns"
        ],

        "num_original_B_turns": filtered[
            "num_original_B_turns"
        ],

        "num_filtered_A_turns": filtered[
            "num_filtered_A_turns"
        ],

        "num_filtered_B_turns": filtered[
            "num_filtered_B_turns"
        ],

        "num_removed_A_backchannels": filtered[
            "num_removed_A_backchannels"
        ],

        "num_removed_B_backchannels": filtered[
            "num_removed_B_backchannels"
        ],

        "removed_A_backchannel_indices": filtered[
            "removed_A_backchannel_indices"
        ],

        "removed_B_backchannel_indices": filtered[
            "removed_B_backchannel_indices"
        ],

        "removed_A_backchannels": filtered[
            "removed_A_backchannels"
        ],

        "removed_B_backchannels": filtered[
            "removed_B_backchannels"
        ],

        "signed_strict_events": events,

        "filtered_turns_A": filtered[
            "filtered_turns_A"
        ],

        "filtered_turns_B": filtered[
            "filtered_turns_B"
        ],
    }

In [ ]:
def add_independent_bc_features(
    source_cases,
    description,
):
    output_cases = []

    for case in tqdm(
        source_cases,
        desc=description,
    ):
        new_case = dict(case)
        features = compute_independent_bc_features(case)

        new_case["independent_bc_temporal_features"] = features
        new_case["filtered_turns_A"] = features["filtered_turns_A"]
        new_case["filtered_turns_B"] = features["filtered_turns_B"]

        output_cases.append(new_case)

    return output_cases


reference_independent_bc_cases = add_independent_bc_features(
    reference_cases,
    "Reference temporal features",
)

inference_independent_bc_cases = add_independent_bc_features(
    inference_cases,
    "Inference temporal features",
)

REFERENCE_FEATURE_CASES_PATH.write_text(
    json.dumps(
        reference_independent_bc_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

INFERENCE_FEATURE_CASES_PATH.write_text(
    json.dumps(
        inference_independent_bc_cases,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Reference feature cases:", len(reference_independent_bc_cases))
print("Inference feature cases:", len(inference_independent_bc_cases))

Reference temporal features:   0%|          | 0/200 [00:00<?, ?it/s]

Inference temporal features:   0%|          | 0/210 [00:00<?, ?it/s]

Reference feature cases: 200
Inference feature cases: 210


## 12. Estimate Frozen Base Statistics for Each Temporal Profile

Using only the 50-conversation reference split, compute separate reference statistics for:

- NORMAL
- LAG `+1 s`
- LAG `+2 s`
- LAG `+3 s`

These statistics describe the local temporal feature distributions and remain frozen during held-out development inference.

In [ ]:
def build_base_reference_row(case):
    features = case["independent_bc_temporal_features"]

    return {
        "case_id": case["case_id"],
        "conversation_id": case["conversation_id"],
        "reference_profile": case["reference_profile"],
        "num_original_A_turns": features["num_original_A_turns"],
        "num_original_B_turns": features["num_original_B_turns"],
        "num_removed_A_backchannels": features["num_removed_A_backchannels"],
        "num_removed_B_backchannels": features["num_removed_B_backchannels"],
        "num_filtered_A_turns": features["num_filtered_A_turns"],
        "num_filtered_B_turns": features["num_filtered_B_turns"],
        "num_offsets": features["num_signed_strict_offsets"],
        "num_negative": features["num_negative_preend_offsets"],
        "num_positive": features["num_positive_postend_offsets"],
        "offset_mean": features["offset_mean_seconds"],
        "offset_median": features["offset_median_seconds"],
        "offset_min": features["offset_min_seconds"],
        "offset_max": features["offset_max_seconds"],
        "offset_p75": features["offset_p75_seconds"],
        "offset_p90": features["offset_p90_seconds"],
        "percent_above_1_5": features["percent_offsets_above_1_5_seconds"],
        "clean_overlap_seconds": features["clean_overlap_seconds"],
        "clean_overlap_percent": features["clean_overlap_percent"],
    }


reference_base_df = pd.DataFrame(
    [
        build_base_reference_row(case)
        for case in reference_independent_bc_cases
    ]
)

PROFILE_ORDER = [
    "NORMAL",
    "LAG_1",
    "LAG_2",
    "LAG_3",
]

reference_base_stats = (
    reference_base_df
    .groupby("reference_profile")
    .mean(numeric_only=True)
    .reindex(PROFILE_ORDER)
)


def base_reference_value(profile, column):
    value = reference_base_stats.loc[profile, column]

    if pd.isna(value):
        return 0.0

    return float(value)


REFERENCE_BASE_STATS_PATH.write_text(
    json.dumps(
        reference_base_stats
        .reset_index()
        .to_dict(orient="records"),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("FROZEN BASE REFERENCE STATISTICS")

display(
    reference_base_stats[
        [
            "num_offsets",
            "offset_mean",
            "offset_median",
            "offset_max",
            "offset_p75",
            "offset_p90",
            "percent_above_1_5",
            "clean_overlap_seconds",
        ]
    ].round(3)
)

assert list(reference_base_stats.index) == PROFILE_ORDER
assert all(
    (reference_base_df["reference_profile"] == profile).sum() == 50
    for profile in PROFILE_ORDER
)

print("Saved frozen base statistics:", REFERENCE_BASE_STATS_PATH)

FROZEN BASE REFERENCE STATISTICS


,num_offsets,offset_mean,offset_median,offset_max,offset_p75,offset_p90,percent_above_1_5,clean_overlap_seconds
reference_profile,,,,,,,,
NORMAL,5.76,0.302,0.262,1.054,0.590,0.825,6.734,6.253
LAG_1,5.22,0.936,0.988,1.717,1.273,1.493,24.133,8.040
LAG_2,4.48,1.333,1.298,2.506,1.866,2.228,51.149,10.452
LAG_3,4.46,1.586,1.516,3.067,2.184,2.712,43.525,12.200


Saved frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json


## 13. Compute the Global Participant-B Correction Shift

Local offsets describe individual handoffs but may not capture an interaction-wide temporal displacement.

The global alignment feature therefore searches for the temporal correction that best aligns Participant B with Participant A over the complete observed interaction.

A negative best correction indicates that Participant B would need to move earlier to improve alignment, which is consistent with a delayed Participant-B timeline.

In [ ]:
# ============================================================
# DIAGNOSTIC FEATURE:
# Estimated global correction shift for Participant B
#
# Input:
#   independently backchannel-filtered A/B turns
#
# No gold label is used during feature computation.
# ============================================================

import numpy as np
import pandas as pd


# Search possible corrections applied to B.
#
# Negative:
#   B must move earlier.
#
# Positive:
#   B must move later.
ALIGNMENT_SHIFT_MIN_SECONDS = -6.0
ALIGNMENT_SHIFT_MAX_SECONDS = 6.0
ALIGNMENT_SHIFT_STEP_SECONDS = 0.10

# Controls how quickly large boundary errors lose score.
ALIGNMENT_OFFSET_SCALE_SECONDS = 1.50


def alignment_sorted_turns(turns):
    return sorted(
        [
            {
                "start": float(turn["start"]),
                "end": float(turn["end"]),
            }
            for turn in turns
            if (
                float(turn["end"])
                > float(turn["start"])
            )
        ],
        key=lambda turn: (
            turn["start"],
            turn["end"],
        ),
    )


def shift_turns_for_alignment(
    turns,
    shift_seconds,
    timeline_duration,
):
    """
    Apply a hypothetical global correction shift to B.

    Turns are clipped to the observed analysis window.
    """

    shift_seconds = float(
        shift_seconds
    )

    timeline_duration = float(
        timeline_duration
    )

    shifted = []

    for turn in alignment_sorted_turns(
        turns
    ):
        new_start = (
            float(turn["start"])
            + shift_seconds
        )

        new_end = (
            float(turn["end"])
            + shift_seconds
        )

        # Entire turn lies before or after the window.
        if new_end <= 0:
            continue

        if new_start >= timeline_duration:
            continue

        new_start = max(
            0.0,
            new_start,
        )

        new_end = min(
            timeline_duration,
            new_end,
        )

        if new_end <= new_start:
            continue

        shifted.append({
            "start": round(
                new_start,
                3,
            ),

            "end": round(
                new_end,
                3,
            ),
        })

    return shifted


def evaluate_bilateral_alignment_for_shift(
    turns_A,
    turns_B,
    shift_seconds,
    timeline_duration,
):
    """
    Evaluate one hypothetical correction shift of B.

    The same strict signed-offset extraction is used in
    both directions:

        A_end -> shifted B_start
        shifted B_end -> A_start

    The alignment score rewards:

    1. More valid bilateral strict events.
    2. Events whose signed offsets are close to zero.
    """

    turns_A = alignment_sorted_turns(
        turns_A
    )

    shifted_B = shift_turns_for_alignment(
        turns_B,
        shift_seconds=shift_seconds,
        timeline_duration=timeline_duration,
    )

    # A finishes -> B begins
    A_to_B_events = (
        duration_only_signed_strict_events(
            turns_A,
            shifted_B,

            max_post_delay_seconds=(
                INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
            ),

            max_prestart_seconds=(
                INDEPENDENT_BC_MAX_PRESTART_SECONDS
            ),
        )
    )

    # B finishes -> A begins
    B_to_A_events = (
        duration_only_signed_strict_events(
            shifted_B,
            turns_A,

            max_post_delay_seconds=(
                INDEPENDENT_BC_MAX_POST_DELAY_SECONDS
            ),

            max_prestart_seconds=(
                INDEPENDENT_BC_MAX_PRESTART_SECONDS
            ),
        )
    )

    A_to_B_offsets = [
        float(
            event["signed_offset_seconds"]
        )
        for event in A_to_B_events
    ]

    B_to_A_offsets = [
        float(
            event["signed_offset_seconds"]
        )
        for event in B_to_A_events
    ]

    all_offsets = np.asarray(
        A_to_B_offsets
        + B_to_A_offsets,
        dtype=float,
    )

    num_events = int(
        len(all_offsets)
    )

    # Maximum possible evidence is approximately one
    # event per turn ending in both directions.
    possible_boundaries = max(
        1,
        len(turns_A)
        + len(shifted_B),
    )

    event_coverage = (
        num_events
        / possible_boundaries
    )

    if num_events == 0:
        return {
            "candidate_shift_seconds": round(
                float(shift_seconds),
                3,
            ),

            "alignment_score": 0.0,

            "num_bilateral_events": 0,
            "num_A_to_B_events": 0,
            "num_B_to_A_events": 0,

            "event_coverage": 0.0,

            "mean_abs_offset_seconds": None,
            "median_abs_offset_seconds": None,

            "mean_signed_offset_seconds": None,
            "median_signed_offset_seconds": None,

            "A_to_B_offsets": [],
            "B_to_A_offsets": [],

            "num_shifted_B_turns": len(
                shifted_B
            ),
        }

    absolute_offsets = np.abs(
        all_offsets
    )

    # Each event receives a score close to 1 when it lies
    # near zero, decreasing smoothly for larger offsets.
    boundary_closeness = float(
        np.mean(
            np.exp(
                -absolute_offsets
                / ALIGNMENT_OFFSET_SCALE_SECONDS
            )
        )
    )

    # A candidate needs both:
    #   good coverage
    #   offsets close to zero
    alignment_score = (
        event_coverage
        * boundary_closeness
    )

    return {
        "candidate_shift_seconds": round(
            float(shift_seconds),
            3,
        ),

        "alignment_score": round(
            float(alignment_score),
            6,
        ),

        "num_bilateral_events": num_events,

        "num_A_to_B_events": len(
            A_to_B_events
        ),

        "num_B_to_A_events": len(
            B_to_A_events
        ),

        "event_coverage": round(
            float(event_coverage),
            6,
        ),

        "mean_abs_offset_seconds": round(
            float(
                np.mean(
                    absolute_offsets
                )
            ),
            3,
        ),

        "median_abs_offset_seconds": round(
            float(
                np.median(
                    absolute_offsets
                )
            ),
            3,
        ),

        "mean_signed_offset_seconds": round(
            float(
                np.mean(
                    all_offsets
                )
            ),
            3,
        ),

        "median_signed_offset_seconds": round(
            float(
                np.median(
                    all_offsets
                )
            ),
            3,
        ),

        "A_to_B_offsets": [
            round(value, 3)
            for value in A_to_B_offsets
        ],

        "B_to_A_offsets": [
            round(value, 3)
            for value in B_to_A_offsets
        ],

        "num_shifted_B_turns": len(
            shifted_B
        ),
    }


def estimate_global_B_correction_shift(
    turns_A,
    turns_B,
    timeline_duration,
):
    """
    Search for the global B correction shift that produces
    the strongest bilateral boundary alignment.

    This function uses no label information.
    """

    candidate_shifts = np.round(
        np.arange(
            ALIGNMENT_SHIFT_MIN_SECONDS,
            (
                ALIGNMENT_SHIFT_MAX_SECONDS
                + ALIGNMENT_SHIFT_STEP_SECONDS / 2
            ),
            ALIGNMENT_SHIFT_STEP_SECONDS,
        ),
        3,
    )

    candidate_rows = [
        evaluate_bilateral_alignment_for_shift(
            turns_A=turns_A,
            turns_B=turns_B,

            shift_seconds=float(
                candidate_shift
            ),

            timeline_duration=(
                timeline_duration
            ),
        )
        for candidate_shift
        in candidate_shifts
    ]

    # Sort using:
    # 1. highest score
    # 2. highest event coverage
    # 3. lowest median absolute error
    # 4. smallest absolute correction as final tie-break
    def candidate_sort_key(row):
        median_abs = row[
            "median_abs_offset_seconds"
        ]

        if median_abs is None:
            median_abs = float("inf")

        return (
            -float(
                row["alignment_score"]
            ),

            -float(
                row["event_coverage"]
            ),

            float(median_abs),

            abs(
                float(
                    row[
                        "candidate_shift_seconds"
                    ]
                )
            ),
        )

    ranked_rows = sorted(
        candidate_rows,
        key=candidate_sort_key,
    )

    best = ranked_rows[0]

    zero_shift_rows = [
        row
        for row in candidate_rows
        if abs(
            float(
                row[
                    "candidate_shift_seconds"
                ]
            )
        ) < 1e-9
    ]

    if zero_shift_rows:
        zero_shift = zero_shift_rows[0]
    else:
        zero_shift = min(
            candidate_rows,
            key=lambda row: abs(
                float(
                    row[
                        "candidate_shift_seconds"
                    ]
                )
            ),
        )

    best_shift = float(
        best[
            "candidate_shift_seconds"
        ]
    )

    score_gain = (
        float(best["alignment_score"])
        - float(
            zero_shift[
                "alignment_score"
            ]
        )
    )

    # Since synthetic lag moves B later, a negative correction
    # estimates how late B appears to be.
    estimated_B_lateness = max(
        0.0,
        -best_shift,
    )

    return {
        "best_B_correction_shift_seconds": round(
            best_shift,
            3,
        ),

        "estimated_B_lateness_seconds": round(
            estimated_B_lateness,
            3,
        ),

        "best_alignment_score": float(
            best["alignment_score"]
        ),

        "zero_shift_alignment_score": float(
            zero_shift[
                "alignment_score"
            ]
        ),

        "alignment_score_gain_vs_zero": round(
            score_gain,
            6,
        ),

        "best_num_bilateral_events": int(
            best[
                "num_bilateral_events"
            ]
        ),

        "best_num_A_to_B_events": int(
            best[
                "num_A_to_B_events"
            ]
        ),

        "best_num_B_to_A_events": int(
            best[
                "num_B_to_A_events"
            ]
        ),

        "best_event_coverage": float(
            best[
                "event_coverage"
            ]
        ),

        "best_mean_abs_offset_seconds": (
            best[
                "mean_abs_offset_seconds"
            ]
        ),

        "best_median_abs_offset_seconds": (
            best[
                "median_abs_offset_seconds"
            ]
        ),

        "best_mean_signed_offset_seconds": (
            best[
                "mean_signed_offset_seconds"
            ]
        ),

        "best_median_signed_offset_seconds": (
            best[
                "median_signed_offset_seconds"
            ]
        ),

        "best_A_to_B_offsets": best[
            "A_to_B_offsets"
        ],

        "best_B_to_A_offsets": best[
            "B_to_A_offsets"
        ],

        "best_shift_at_search_boundary": bool(
            abs(
                best_shift
                - ALIGNMENT_SHIFT_MIN_SECONDS
            )
            < 1e-9
            or abs(
                best_shift
                - ALIGNMENT_SHIFT_MAX_SECONDS
            )
            < 1e-9
        ),

        # Keep all candidates for later inspection.
        "alignment_shift_curve": (
            candidate_rows
        ),
    }

## 14. Attach Global-Shift Features and Estimate Frozen Global References

Compute and cache the global correction features for all reference and development cases.

Separate frozen global-shift statistics are then estimated from the 50-conversation reference split for each temporal profile.

In [ ]:
MIXED_EXPERIMENT_VERSION = (
    "guided_independent_bc_signed_offsets_"
    "plus_global_shift_mixed_lag_v1"
)

MIXED_PROMPT_VERSION = (
    f"{MIXED_EXPERIMENT_VERSION}"
    f"_duration{INDEPENDENT_BC_MAX_DURATION_SECONDS}"
    f"_overlap{INDEPENDENT_BC_MIN_OVERLAP_FRACTION}"
    f"_threshold{INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS}"
    f"_postmax{INDEPENDENT_BC_MAX_POST_DELAY_SECONDS}"
    f"_premax{INDEPENDENT_BC_MAX_PRESTART_SECONDS}"
    f"_shiftmin{ALIGNMENT_SHIFT_MIN_SECONDS}"
    f"_shiftmax{ALIGNMENT_SHIFT_MAX_SECONDS}"
    f"_shiftstep{ALIGNMENT_SHIFT_STEP_SECONDS}"
    f"_scale{ALIGNMENT_OFFSET_SCALE_SECONDS}"
    "_frozen50_heldout100"
)

GLOBAL_SHIFT_COMPACT_KEYS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "best_alignment_score",
    "zero_shift_alignment_score",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_num_A_to_B_events",
    "best_num_B_to_A_events",
    "best_event_coverage",
    "best_mean_abs_offset_seconds",
    "best_median_abs_offset_seconds",
    "best_mean_signed_offset_seconds",
    "best_median_signed_offset_seconds",
    "best_shift_at_search_boundary",
]

if GLOBAL_SHIFT_CACHE_PATH.exists():
    try:
        global_shift_feature_cache = json.loads(
            GLOBAL_SHIFT_CACHE_PATH.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(global_shift_feature_cache, dict):
            global_shift_feature_cache = {}

    except Exception as exc:
        print("Could not load global-shift cache:", exc)
        global_shift_feature_cache = {}
else:
    global_shift_feature_cache = {}

all_feature_cases = (
    reference_independent_bc_cases
    + inference_independent_bc_cases
)

for case in tqdm(
    all_feature_cases,
    desc="Global B-correction features",
):
    case_id = case["case_id"]

    if case_id in global_shift_feature_cache:
        continue

    full_shift_features = estimate_global_B_correction_shift(
        turns_A=case["filtered_turns_A"],
        turns_B=case["filtered_turns_B"],
        timeline_duration=case["analysis_duration"],
    )

    global_shift_feature_cache[case_id] = {
        key: full_shift_features[key]
        for key in GLOBAL_SHIFT_COMPACT_KEYS
    }

    GLOBAL_SHIFT_CACHE_PATH.write_text(
        json.dumps(
            global_shift_feature_cache,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def attach_global_shift_features(source_cases):
    output_cases = []

    for case in source_cases:
        new_case = dict(case)
        new_case["global_alignment_shift_features"] = (
            global_shift_feature_cache[case["case_id"]]
        )
        output_cases.append(new_case)

    return output_cases


reference_shift_cases = attach_global_shift_features(
    reference_independent_bc_cases
)

inference_shift_cases = attach_global_shift_features(
    inference_independent_bc_cases
)

reference_shift_rows = []

for case in reference_shift_cases:
    reference_shift_rows.append({
        "case_id": case["case_id"],
        "conversation_id": case["conversation_id"],
        "reference_profile": case["reference_profile"],
        **case["global_alignment_shift_features"],
    })

reference_shift_df = pd.DataFrame(reference_shift_rows)

SHIFT_REFERENCE_CORE_COLUMNS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
]

SHIFT_REFERENCE_RELIABILITY_COLUMNS = [
    "best_num_bilateral_events",
    "best_event_coverage",
]

reference_shift_stats = (
    reference_shift_df
    .groupby("reference_profile")
    .agg({
        **{
            column: ["mean", "median"]
            for column in SHIFT_REFERENCE_CORE_COLUMNS
        },
        **{
            column: ["mean"]
            for column in SHIFT_REFERENCE_RELIABILITY_COLUMNS
        },
    })
    .reindex(PROFILE_ORDER)
)


def shift_reference_value(
    profile,
    column,
    statistic="mean",
):
    value = reference_shift_stats.loc[
        profile,
        (column, statistic),
    ]

    if pd.isna(value):
        return 0.0

    return float(value)


shift_stats_records = []

for profile in PROFILE_ORDER:
    record = {
        "reference_profile": profile
    }

    for column in (
        SHIFT_REFERENCE_CORE_COLUMNS
        + SHIFT_REFERENCE_RELIABILITY_COLUMNS
    ):
        statistics = (
            ["mean", "median"]
            if column in SHIFT_REFERENCE_CORE_COLUMNS
            else ["mean"]
        )

        for statistic in statistics:
            record[f"{column}__{statistic}"] = shift_reference_value(
                profile,
                column,
                statistic,
            )

    shift_stats_records.append(record)

REFERENCE_SHIFT_STATS_PATH.write_text(
    json.dumps(
        shift_stats_records,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("FROZEN GLOBAL-SHIFT REFERENCE STATISTICS")
display(reference_shift_stats.round(3))

print("Mixed prompt version:", MIXED_PROMPT_VERSION)
print("Global-shift cache entries:", len(global_shift_feature_cache))
print("Saved frozen shift statistics:", REFERENCE_SHIFT_STATS_PATH)

Global B-correction features:   0%|          | 0/410 [00:00<?, ?it/s]

FROZEN GLOBAL-SHIFT REFERENCE STATISTICS


best_B_correction_shift_seconds         \
                                             mean median   
reference_profile                                          
NORMAL                                     -0.230   -0.0   
LAG_1                                      -0.680   -0.9   
LAG_2                                      -1.100   -1.9   
LAG_3                                      -2.034   -2.9   

                  estimated_B_lateness_seconds         \
                                          mean median   
reference_profile                                       
NORMAL                                   0.500    0.0   
LAG_1                                    1.084    0.9   
LAG_2                                    1.668    1.9   
LAG_3                                    2.464    2.9   

                  alignment_score_gain_vs_zero         \
                                          mean median   
reference_profile                                       
NORMAL                                   0.023  0.009   
LAG_1                                    0.105  0.095   
LAG_2                                    0.188  0.185   
LAG_3                                    0.207  0.179   

                  best_num_bilateral_events best_event_coverage  
                                       mean                mean  
reference_profile                                                
NORMAL                                11.94               0.595  
LAG_1                                 10.76               0.560  
LAG_2                                 10.78               0.562  
LAG_3                                 10.62               0.551

Mixed prompt version: guided_independent_bc_signed_offsets_plus_global_shift_mixed_lag_v1_duration1.0_overlap0.8_threshold1.5_postmax6.0_premax2.0_shiftmin-6.0_shiftmax6.0_shiftstep0.1_scale1.5_frozen50_heldout100
Global-shift cache entries: 410
Saved frozen shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json


## 15. Build the Development Temporal Reasoning Prompt

The temporal reasoner receives the complete development evidence packet:

- independently filtered Participant-A and Participant-B turns;
- cleaned overlap information;
- signed strict local offsets and their summary statistics;
- global alignment-shift features;
- frozen NORMAL and synthetic-LAG reference patterns.

The current case's gold label, source conversation ID, variant name, and applied synthetic delay are not supplied to Qwen.

This richer prompt is part of the **development pipeline** and should not be confused with the later reduced prompt used for the thesis-reported isolated temporal result.

In [ ]:
def value_seconds(value):
    if value is None:
        return "Not available"

    return f"{float(value):.2f} seconds"


def value_percent(value):
    if value is None:
        return "Not available"

    return f"{float(value):.1f}%"


PROFILE_TITLES = {
    "NORMAL": "Typical NORMAL cases",
    "LAG_1": "Typical LAG +1 second cases",
    "LAG_2": "Typical LAG +2 second cases",
    "LAG_3": "Typical LAG +3 second cases",
}


def build_base_reference_section(profile):
    title = PROFILE_TITLES[profile]

    return f"""
{title}:

- Number of signed offsets around {base_reference_value(profile, "num_offsets"):.2f}.
- Mean signed offset around {base_reference_value(profile, "offset_mean"):.2f} seconds.
- Median signed offset around {base_reference_value(profile, "offset_median"):.2f} seconds.
- Maximum signed offset around {base_reference_value(profile, "offset_max"):.2f} seconds.
- P75 around {base_reference_value(profile, "offset_p75"):.2f} seconds.
- P90 around {base_reference_value(profile, "offset_p90"):.2f} seconds.
- Approximately {base_reference_value(profile, "percent_above_1_5"):.1f}% of offsets are above 1.5 seconds.
- Filtered overlap is around {base_reference_value(profile, "clean_overlap_seconds"):.2f} seconds.
""".strip()


def build_global_reference_section(profile):
    title = PROFILE_TITLES[profile]

    return f"""
{title}:

- Mean best B correction shift is around {shift_reference_value(profile, "best_B_correction_shift_seconds", "mean"):.2f} seconds.
- Median best B correction shift is around {shift_reference_value(profile, "best_B_correction_shift_seconds", "median"):.2f} seconds.
- Mean estimated B lateness is around {shift_reference_value(profile, "estimated_B_lateness_seconds", "mean"):.2f} seconds.
- Median estimated B lateness is around {shift_reference_value(profile, "estimated_B_lateness_seconds", "median"):.2f} seconds.
- Mean alignment score gain versus zero shift is around {shift_reference_value(profile, "alignment_score_gain_vs_zero", "mean"):.3f}.
- Median alignment score gain versus zero shift is around {shift_reference_value(profile, "alignment_score_gain_vs_zero", "median"):.3f}.
- Mean number of bilateral alignment events is around {shift_reference_value(profile, "best_num_bilateral_events", "mean"):.2f}.
- Mean bilateral event coverage is around {100.0 * shift_reference_value(profile, "best_event_coverage", "mean"):.2f}%.
""".strip()


BASE_REFERENCE_TEXT = "\n\n".join(
    build_base_reference_section(profile)
    for profile in PROFILE_ORDER
)

GLOBAL_REFERENCE_TEXT = "\n\n".join(
    build_global_reference_section(profile)
    for profile in PROFILE_ORDER
)


MIXED_LAG_PROMPT_TEMPLATE = '''
You are evaluating temporal coordination between two participants in the same dyadic conversation.

You receive:

1. Independently backchannel-filtered speech turns for Participant A.
2. Independently backchannel-filtered speech turns for Participant B.
3. Overlap features calculated after independent backchannel filtering.
4. A signed strict A_end-to-B_start offset list.
5. Distribution summaries calculated from that exact signed list.
6. Global B-correction alignment-shift features.

TURN FORMAT

- Every turn is represented as [start_time, end_time] in seconds.
- Participant A and Participant B use the same aligned timeline.
- A merged turn was removed only when it lasted at most {max_short_turn_seconds:.1f} seconds and at least {backchannel_overlap_percent:.0f}% of its duration overlapped the other participant.
- Duration and overlap were calculated independently from the observed A/B turns of each case.
- The filtering did not use the NORMAL or LAG label.
- The removed turns are operational backchannel-like candidates and are not semantically verified backchannels.

SIGNED STRICT A_END-TO-B_START OFFSETS

Each value is calculated as:

B_start minus A_end

Positive value:

- Participant B starts after Participant A ends.
- Positive offsets are retained up to {max_post_delay_seconds:.1f} seconds.
- Participant A must not start another turn before or at B_start.

Negative value:

- Participant B starts shortly before Participant A ends.
- Participant B must still be speaking when Participant A ends.
- Negative offsets are retained only when B starts during the current A turn and no more than {max_prestart_seconds:.1f} seconds before A_end.

Examples:

- -0.60 seconds means B starts 0.60 seconds before A ends.
- 0.00 seconds means an immediate boundary transition.
- 2.50 seconds means B starts 2.50 seconds after A ends.

The supplied list therefore describes local handoff timing on both sides of A_end.

SOFT REFERENCE PATTERNS FROM THE FROZEN 50-CONVERSATION REFERENCE SET

{base_reference_text}

These are soft exploratory patterns and not hard thresholds.
Natural variation exists in every profile.

DECISION GUIDANCE

- Use the complete signed offset distribution.
- Consider the full list, number of values, mean, median, maximum, P75, P90, and percentage above 1.5 seconds together.
- Compare the current case with the NORMAL profile and with each separate LAG +1, LAG +2, and LAG +3 profile.
- A later shift of Participant B tends to move signed offsets toward more positive values.
- Several elevated positive offsets support LAG.
- Elevated mean, median, P75, or P90 support LAG.
- Negative or near-zero offsets represent smooth or slightly overlapping handoffs and generally support NORMAL when they occur consistently.
- A single negative value does not automatically indicate NORMAL.
- A single positive maximum does not automatically indicate LAG.
- Do not decide from only one statistic.
- Do not require every value to be positive or large for LAG.
- A LAG case may still contain negative or short positive values.
- The final output remains binary. Do not predict the delay magnitude.

LIMITED EVIDENCE

- When the signed offset list is empty, the offset statistics are unavailable.
- When the list contains only one or two values, the statistics are based on limited evidence.
- In these cases, reduce confidence and use the filtered overlap and raw turn structure cautiously.
- Do not classify LAG solely because of one large positive offset.

FILTERED OVERLAP

- Filtered overlap is secondary supporting evidence.
- Elevated overlap strengthens a LAG decision when the signed offset distribution is also abnormal.
- Overlap alone must not determine the label.

GLOBAL B-CORRECTION REFERENCE PATTERNS

The correction shift is a hypothetical global shift applied only during diagnostic alignment search.

The filtered Participant B turns supplied for classification are not changed.

{global_reference_text}

GLOBAL B-CORRECTION INTERPRETATION

- The best B correction shift is the global temporal correction that maximizes bilateral A-to-B and B-to-A boundary alignment.
- A negative correction means Participant B would need to move earlier to align better with Participant A.
- A correction near -1 second is compatible with an approximate +1 second Participant B delay.
- A correction near -2 seconds is compatible with an approximate +2 second Participant B delay.
- A correction near -3 seconds is compatible with an approximate +3 second Participant B delay.
- Estimated B lateness is max(0, -best correction shift).
- A correction near zero together with a very small alignment gain generally supports NORMAL.
- A clearly negative correction together with a meaningful alignment gain supports LAG.
- The correction value must not be used alone.
- The number of bilateral events and event coverage indicate how much evidence supports the correction estimate.
- A large correction based on very few events or very low coverage is weak evidence.
- Use the global-shift evidence together with the signed strict offset distribution and filtered overlap.

LAG GENERATION

- A LAG case may contain Participant B shifted later by approximately 1, 2, or 3 seconds.
- The delay magnitude, if any, is not provided for the current case.
- The signed offset list does not need to increase uniformly.
- Shifting Participant B can change which strict handoff events remain valid.

CLASSIFY AS NORMAL WHEN

- The signed offset distribution is predominantly negative, near zero, or short positive.
- The mean, median, P75, and P90 are more compatible with the NORMAL reference pattern than with the three LAG profiles.
- Values above 1.5 seconds are absent, rare, or isolated.
- A large maximum is not supported by the rest of the distribution.
- The global correction is near zero or is weakly supported.

CLASSIFY AS LAG WHEN

- The signed offset distribution is shifted toward positive values.
- Several offsets are elevated.
- Or the mean, median, P75, or P90 are more compatible with at least one of the separate LAG profiles.
- A meaningful percentage of offsets above 1.5 seconds supports LAG.
- A reliable negative global correction and meaningful alignment gain support LAG.
- Filtered overlap may provide secondary supporting evidence.

Use only the supplied temporal information.
Do not invent transcript content.
Do not use semantic assumptions.
Return one label even when the evidence is mixed.

Timeline duration:
{duration:.2f} seconds

Participant A filtered turns:
{turns_A}

Participant B filtered turns:
{turns_B}

DERIVED TEMPORAL FEATURES

Filtered overlap seconds:
{clean_overlap_seconds:.2f}

Filtered overlap percentage:
{clean_overlap_percent:.2f}%

Signed strict A_end-to-B_start offset list:
{offset_list}

Number of signed strict offsets:
{num_offsets}

Mean signed offset:
{offset_mean}

Median signed offset:
{offset_median}

Maximum signed offset:
{offset_max}

P75 signed offset:
{offset_p75}

P90 signed offset:
{offset_p90}

Number of offsets above {delay_threshold:.1f} seconds:
{num_above_threshold}

Percentage of offsets above {delay_threshold:.1f} seconds:
{percent_above_threshold}

GLOBAL ALIGNMENT-SHIFT FEATURES

Best B correction shift:
{best_B_correction_shift:.2f} seconds

Estimated B lateness:
{estimated_B_lateness:.2f} seconds

Alignment score gain versus zero shift:
{alignment_score_gain:.3f}

Number of bilateral events supporting the best shift:
{best_num_bilateral_events}

Bilateral event coverage:
{best_event_coverage_percent:.2f}%

Return ONLY valid JSON:
{{
  "label": "NORMAL or LAG",
  "confidence": 0.0,
  "reason": "brief explanation based on the signed offset distribution, global alignment-shift evidence, and filtered overlap"
}}
'''.strip()


def build_mixed_lag_prompt(case):
    temporal = case["independent_bc_temporal_features"]
    shift_features = case["global_alignment_shift_features"]

    return MIXED_LAG_PROMPT_TEMPLATE.format(
        max_short_turn_seconds=INDEPENDENT_BC_MAX_DURATION_SECONDS,
        backchannel_overlap_percent=100.0 * INDEPENDENT_BC_MIN_OVERLAP_FRACTION,
        max_post_delay_seconds=INDEPENDENT_BC_MAX_POST_DELAY_SECONDS,
        max_prestart_seconds=INDEPENDENT_BC_MAX_PRESTART_SECONDS,
        base_reference_text=BASE_REFERENCE_TEXT,
        global_reference_text=GLOBAL_REFERENCE_TEXT,
        duration=float(case["analysis_duration"]),
        turns_A=compact_turns_text(case["filtered_turns_A"]),
        turns_B=compact_turns_text(case["filtered_turns_B"]),
        clean_overlap_seconds=float(temporal["clean_overlap_seconds"]),
        clean_overlap_percent=float(temporal["clean_overlap_percent"]),
        offset_list=json.dumps(
            temporal["signed_strict_offsets_seconds"],
            ensure_ascii=False,
            separators=(",", ":"),
        ),
        num_offsets=int(temporal["num_signed_strict_offsets"]),
        offset_mean=value_seconds(temporal["offset_mean_seconds"]),
        offset_median=value_seconds(temporal["offset_median_seconds"]),
        offset_max=value_seconds(temporal["offset_max_seconds"]),
        offset_p75=value_seconds(temporal["offset_p75_seconds"]),
        offset_p90=value_seconds(temporal["offset_p90_seconds"]),
        delay_threshold=INDEPENDENT_BC_DELAY_THRESHOLD_SECONDS,
        num_above_threshold=int(temporal["num_offsets_above_1_5_seconds"]),
        percent_above_threshold=value_percent(
            temporal["percent_offsets_above_1_5_seconds"]
        ),
        best_B_correction_shift=float(
            shift_features["best_B_correction_shift_seconds"]
        ),
        estimated_B_lateness=float(
            shift_features["estimated_B_lateness_seconds"]
        ),
        alignment_score_gain=float(
            shift_features["alignment_score_gain_vs_zero"]
        ),
        best_num_bilateral_events=int(
            shift_features["best_num_bilateral_events"]
        ),
        best_event_coverage_percent=(
            100.0 * float(shift_features["best_event_coverage"])
        ),
    )


example_mixed_prompt = build_mixed_lag_prompt(
    inference_shift_cases[0]
)

print(example_mixed_prompt[:14000])
print("\nPrompt characters:", len(example_mixed_prompt))


You are evaluating temporal coordination between two participants in the same dyadic conversation.

You receive:

1. Independently backchannel-filtered speech turns for Participant A.
2. Independently backchannel-filtered speech turns for Participant B.
3. Overlap features calculated after independent backchannel filtering.
4. A signed strict A_end-to-B_start offset list.
5. Distribution summaries calculated from that exact signed list.
6. Global B-correction alignment-shift features.

TURN FORMAT

- Every turn is represented as [start_time, end_time] in seconds.
- Participant A and Participant B use the same aligned timeline.
- A merged turn was removed only when it lasted at most 1.0 seconds and at least 80% of its duration overlapped the other participant.
- Duration and overlap were calculated independently from the observed A/B turns of each case.
- The filtering did not use the NORMAL or LAG label.
- The removed turns are operational backchannel-like candidates and are not semant

## 16. Verify Split Isolation and Absence of Label Leakage

Before inference, the notebook verifies that:

- reference and inference conversation IDs are disjoint;
- the expected frozen reference profiles are present;
- the current-case gold label and synthetic delay are absent from the prompt;
- the model output remains binary `NORMAL` versus `LAG`.

In [ ]:
assert len(reference_shift_cases) == 200
assert len(inference_shift_cases) == 210
assert reference_ids.isdisjoint(inference_ids)

assert "Typical NORMAL cases" in example_mixed_prompt
assert "Typical LAG +1 second cases" in example_mixed_prompt
assert "Typical LAG +2 second cases" in example_mixed_prompt
assert "Typical LAG +3 second cases" in example_mixed_prompt

assert (
    "The delay magnitude, if any, is not provided for the current case."
    in example_mixed_prompt
)

assert '"label": "NORMAL or LAG"' in example_mixed_prompt
assert "gold_label" not in example_mixed_prompt
assert "conversation_id" not in example_mixed_prompt
assert "reference_profile" not in example_mixed_prompt
assert "variant" not in example_mixed_prompt

print("Verified: reference and inference conversation IDs are disjoint.")
print("Verified: frozen statistics contain four separate profiles.")
print("Verified: current case label, variant, ID, and applied delay are not supplied to Qwen.")
print("Verified: Qwen output remains binary NORMAL vs LAG.")
print("Held-out cases ready for inference:", len(inference_shift_cases))

Verified: reference and inference conversation IDs are disjoint.
Verified: frozen statistics contain four separate profiles.
Verified: current case label, variant, ID, and applied delay are not supplied to Qwen.
Verified: Qwen output remains binary NORMAL vs LAG.
Held-out cases ready for inference: 210


## 17. Run the Mixed-Delay Development Experiment

Run Qwen inference over the held-out development cases with per-case caching.

This first development experiment includes NORMAL together with LAG `+1 s`, `+2 s`, and `+3 s` cases.

In [ ]:
if RESULTS_PATH.exists():
    try:
        mixed_lag_results = json.loads(
            RESULTS_PATH.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(mixed_lag_results, list):
            mixed_lag_results = []

    except Exception as exc:
        print("Could not load previous results:", exc)
        mixed_lag_results = []
else:
    mixed_lag_results = []

compatible_done_case_ids = {
    result["case_id"]
    for result in mixed_lag_results
    if result.get("mixed_prompt_version") == MIXED_PROMPT_VERSION
}

inference_cases_to_run = inference_shift_cases

if PILOT_MAX_CASES is not None:
    inference_cases_to_run = inference_cases_to_run[:PILOT_MAX_CASES]

print("Cached compatible results:", len(compatible_done_case_ids))
print("Cases requested:", len(inference_cases_to_run))
print(
    "Cases still missing:",
    sum(
        case["case_id"] not in compatible_done_case_ids
        for case in inference_cases_to_run
    ),
)

for case in tqdm(
    inference_cases_to_run,
    desc="Qwen NORMAL vs mixed LAG",
):
    if case["case_id"] in compatible_done_case_ids:
        continue

    prompt = build_mixed_lag_prompt(case)

    raw_output = qwen_text_only(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    parsed = extract_json_from_text(raw_output)

    result = {
        "case_id": case["case_id"],
        "conversation_id": case["conversation_id"],
        "participant_A_id": case["participant_A_id"],
        "participant_B_id": case["participant_B_id"],
        # Stored only for evaluation; not inserted into the prompt.
        "variant": case["variant"],
        "reference_profile": case["reference_profile"],
        "gold_label": case["gold_label"],
        "lag_seconds": case["lag_seconds"],
        "mixed_prompt_version": MIXED_PROMPT_VERSION,
        "analysis_duration": case["analysis_duration"],
        "filtered_turns_A": case["filtered_turns_A"],
        "filtered_turns_B": case["filtered_turns_B"],
        "independent_bc_temporal_features": case[
            "independent_bc_temporal_features"
        ],
        "global_alignment_shift_features": case[
            "global_alignment_shift_features"
        ],
        "prompt": prompt,
        "raw_output": raw_output,
        "parsed": parsed,
    }

    mixed_lag_results.append(result)
    compatible_done_case_ids.add(case["case_id"])

    RESULTS_PATH.write_text(
        json.dumps(
            mixed_lag_results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

print("Total stored results:", len(mixed_lag_results))
print("Saved:", RESULTS_PATH)

Cached compatible results: 0
Cases requested: 210
Cases still missing: 210


Qwen NORMAL vs mixed LAG:   0%|          | 0/210 [00:00<?, ?it/s]

Total stored results: 210
Saved: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/qwen_mixed_lag_binary_results.json


## 18. Evaluate the Full Mixed-Delay Development Experiment

Evaluate overall binary NORMAL-versus-LAG performance and inspect recall separately for each synthetic delay magnitude.

These results are **development diagnostics**, not the final isolated temporal result reported in the thesis.

In [ ]:
# ============================================================
# Normalize Qwen prediction labels
# ============================================================

def normalize_independent_bc_prediction(parsed):
    if not isinstance(parsed, dict):
        return None

    label = str(
        parsed.get("label", "")
    ).strip().upper()

    aliases = {
        "NORMAL": "NORMAL",
        "LAG": "LAG",
        "ANOMALOUS": "LAG",
        "ANOMALY": "LAG",
        "LAG ANOMALY": "LAG",
        "DELAY": "LAG",
        "DELAYED": "LAG",
    }

    return aliases.get(label)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

requested_case_ids = {
    case["case_id"]
    for case in inference_cases_to_run
}

compatible_results = {}

for result in mixed_lag_results:
    if result.get("mixed_prompt_version") != MIXED_PROMPT_VERSION:
        continue

    if result["case_id"] not in requested_case_ids:
        continue

    compatible_results[result["case_id"]] = result

rows = []

for result in compatible_results.values():
    parsed = result.get("parsed", {})
    predicted_label = normalize_independent_bc_prediction(parsed)

    temporal = result["independent_bc_temporal_features"]
    shift_features = result["global_alignment_shift_features"]

    rows.append({
        "case_id": result["case_id"],
        "conversation_id": result["conversation_id"],
        "reference_profile": result["reference_profile"],
        "lag_seconds": result["lag_seconds"],
        "gold_label": result["gold_label"],
        "predicted_label": predicted_label,
        "confidence": (
            parsed.get("confidence")
            if isinstance(parsed, dict)
            else None
        ),
        "reason": (
            parsed.get("reason")
            if isinstance(parsed, dict)
            else None
        ),
        "parse_error": predicted_label is None,
        "offset_list": json.dumps(
            temporal["signed_strict_offsets_seconds"],
            ensure_ascii=False,
        ),
        "num_offsets": temporal["num_signed_strict_offsets"],
        "offset_mean": temporal["offset_mean_seconds"],
        "offset_median": temporal["offset_median_seconds"],
        "offset_max": temporal["offset_max_seconds"],
        "offset_p75": temporal["offset_p75_seconds"],
        "offset_p90": temporal["offset_p90_seconds"],
        "percent_above_1_5": temporal[
            "percent_offsets_above_1_5_seconds"
        ],
        "clean_overlap_seconds": temporal["clean_overlap_seconds"],
        "clean_overlap_percent": temporal["clean_overlap_percent"],
        "best_B_correction_shift_seconds": shift_features[
            "best_B_correction_shift_seconds"
        ],
        "estimated_B_lateness_seconds": shift_features[
            "estimated_B_lateness_seconds"
        ],
        "alignment_score_gain_vs_zero": shift_features[
            "alignment_score_gain_vs_zero"
        ],
        "best_num_bilateral_events": shift_features[
            "best_num_bilateral_events"
        ],
        "best_event_coverage": shift_features["best_event_coverage"],
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_CSV_PATH, index=False)

print("Evaluated rows:", len(results_df))
print("Parse/label errors:", int(results_df["parse_error"].sum()))
print("Saved CSV:", RESULTS_CSV_PATH)

valid_df = results_df[
    results_df["predicted_label"].isin(["NORMAL", "LAG"])
].copy()

if len(valid_df) == 0:
    raise RuntimeError("No valid NORMAL/LAG predictions were parsed.")

print(
    "\nOverall accuracy:",
    accuracy_score(
        valid_df["gold_label"],
        valid_df["predicted_label"],
    ),
)

print(
    "Overall balanced accuracy:",
    balanced_accuracy_score(
        valid_df["gold_label"],
        valid_df["predicted_label"],
    ),
)

print("\nOverall classification report:")
print(
    classification_report(
        valid_df["gold_label"],
        valid_df["predicted_label"],
        labels=["NORMAL", "LAG"],
        zero_division=0,
    )
)

overall_cm = confusion_matrix(
    valid_df["gold_label"],
    valid_df["predicted_label"],
    labels=["NORMAL", "LAG"],
)

overall_confusion_df = pd.DataFrame(
    overall_cm,
    index=["Gold NORMAL", "Gold LAG"],
    columns=["Pred NORMAL", "Pred LAG"],
)

print("\nOVERALL CONFUSION MATRIX")
display(overall_confusion_df)

profile_prediction_table = (
    pd.crosstab(
        valid_df["reference_profile"],
        valid_df["predicted_label"],
    )
    .reindex(PROFILE_ORDER, fill_value=0)
    .reindex(columns=["NORMAL", "LAG"], fill_value=0)
)

profile_prediction_table.columns = [
    "Pred NORMAL",
    "Pred LAG",
]

print("\nPREDICTIONS BY TRUE PROFILE")
display(profile_prediction_table)

profile_rows = []

for profile in PROFILE_ORDER:
    profile_df = valid_df[
        valid_df["reference_profile"] == profile
    ]

    correct_label = (
        "NORMAL"
        if profile == "NORMAL"
        else "LAG"
    )

    profile_rows.append({
        "true_profile": profile,
        "num_cases": len(profile_df),
        "correct_predictions": int(
            (
                profile_df["predicted_label"]
                == correct_label
            ).sum()
        ),
        "profile_accuracy_or_recall": (
            (
                profile_df["predicted_label"]
                == correct_label
            ).mean()
            if len(profile_df)
            else None
        ),
    })

profile_metrics_df = pd.DataFrame(profile_rows)

print("\nPROFILE-SPECIFIC PERFORMANCE")
display(profile_metrics_df)

print("\nExpected full-run counts:")
print({
    "NORMAL": 100,
    "LAG_1": 30,
    "LAG_2": 40,
    "LAG_3": 40,
})

Evaluated rows: 210
Parse/label errors: 0
Saved CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/qwen_mixed_lag_binary_results.csv

Overall accuracy: 0.7714285714285715
Overall balanced accuracy: 0.7681818181818181

Overall classification report:
              precision    recall  f1-score   support

      NORMAL       0.80      0.70      0.74       100
         LAG       0.75      0.84      0.79       110

    accuracy                           0.77       210
   macro avg       0.77      0.77      0.77       210
weighted avg       0.77      0.77      0.77       210


OVERALL CONFUSION MATRIX


,Pred NORMAL,Pred LAG
Gold NORMAL,70,30
Gold LAG,18,92



PREDICTIONS BY TRUE PROFILE


,Pred NORMAL,Pred LAG
reference_profile,,
NORMAL,70,30
LAG_1,9,21
LAG_2,4,36
LAG_3,5,35



PROFILE-SPECIFIC PERFORMANCE


,true_profile,num_cases,correct_predictions,profile_accuracy_or_recall
0,NORMAL,100,70,0.700
1,LAG_1,30,21,0.700
2,LAG_2,40,36,0.900
3,LAG_3,40,35,0.875



Expected full-run counts:
{'NORMAL': 100, 'LAG_1': 30, 'LAG_2': 40, 'LAG_3': 40}


## Development Result: Mixed `+1 s / +2 s / +3 s` LAG

The richer temporal packet achieves different sensitivity across perturbation magnitudes. The profile-specific evaluation is useful for understanding how detectability changes as the synthetic delay becomes larger.

This stage was used to study and refine the temporal representation rather than to define the final reported configuration.

# Development Follow-Up: NORMAL vs LAG `+2 s` and `+3 s` Only

The same development pipeline is now re-evaluated after removing the `+1 s` profile.

Nothing else in the temporal feature extraction is changed:

- the same held-out conversations are used;
- the same frozen 50-conversation reference statistics are retained;
- the same independently filtered turns are used;
- the same overlap, signed-offset, and global-shift features are used;
- the output remains binary `NORMAL` versus `LAG`.

This follow-up tests the richer temporal representation on the more detectable `+2 s` and `+3 s` perturbations before the subsequent feature-reduction stage.

In [ ]:
# ============================================================
# NEW EXPERIMENT:
# NORMAL vs LAG +2 s / LAG +3 s ONLY
#
# Same held-out conversations, same deterministic features,
# same frozen 50-conversation statistics and same Qwen pipeline.
#
# The only change:
# LAG_1 is removed from the inference set and from the prompt.
# ============================================================

LAG23_PROFILE_ORDER = [
    "NORMAL",
    "LAG_2",
    "LAG_3",
]

LAG23_PROMPT_VERSION = (
    f"{MIXED_PROMPT_VERSION}"
    "_NORMAL_vs_LAG2_LAG3_only_v1"
)

LAG23_RESULTS_PATH = (
    OUT_DIR
    / "qwen_normal_vs_lag2_lag3_binary_results.json"
)

LAG23_RESULTS_CSV_PATH = (
    OUT_DIR
    / "qwen_normal_vs_lag2_lag3_binary_results.csv"
)


# ============================================================
# Keep the same held-out cases, excluding only LAG_1
# ============================================================

lag23_inference_cases = [
    case
    for case in inference_shift_cases
    if case["reference_profile"] in LAG23_PROFILE_ORDER
]

# ============================================================
# Verify the cases that already exist in memory
# ============================================================

actual_lag23_counts = (
    pd.Series(
        [
            case["reference_profile"]
            for case in lag23_inference_cases
        ]
    )
    .value_counts()
    .to_dict()
)

expected_lag23_counts = {
    "NORMAL": 100,
    "LAG_2": 40,
    "LAG_3": 40,
}

print("Actual LAG23 counts:")
print(actual_lag23_counts)

assert actual_lag23_counts == expected_lag23_counts, (
    f"Expected {expected_lag23_counts}, "
    f"but found {actual_lag23_counts}"
)

assert all(
    case["reference_profile"] != "LAG_1"
    for case in lag23_inference_cases
)

print("Cases for the new experiment:")
print(actual_lag23_counts)
print("Total:", len(lag23_inference_cases))


# ============================================================
# Use the same frozen statistics from the original 50
# conversations, but expose only NORMAL, LAG_2 and LAG_3
# ============================================================

LAG23_BASE_REFERENCE_TEXT = "\n\n".join(
    build_base_reference_section(profile)
    for profile in LAG23_PROFILE_ORDER
)

LAG23_GLOBAL_REFERENCE_TEXT = "\n\n".join(
    build_global_reference_section(profile)
    for profile in LAG23_PROFILE_ORDER
)


# ============================================================
# Start from the exact existing prompt template and remove
# only the LAG +1 references
# ============================================================

LAG23_PROMPT_TEMPLATE = MIXED_LAG_PROMPT_TEMPLATE

LAG23_PROMPT_TEMPLATE = LAG23_PROMPT_TEMPLATE.replace(
    (
        "- Compare the current case with the NORMAL profile "
        "and with each separate LAG +1, LAG +2, and LAG +3 profile."
    ),
    (
        "- Compare the current case with the NORMAL profile "
        "and with the separate LAG +2 and LAG +3 profiles."
    ),
)

LAG23_PROMPT_TEMPLATE = LAG23_PROMPT_TEMPLATE.replace(
    (
        "- A correction near -1 second is compatible with an "
        "approximate +1 second Participant B delay.\n"
    ),
    "",
)

LAG23_PROMPT_TEMPLATE = LAG23_PROMPT_TEMPLATE.replace(
    (
        "- A LAG case may contain Participant B shifted later "
        "by approximately 1, 2, or 3 seconds."
    ),
    (
        "- A LAG case may contain Participant B shifted later "
        "by approximately 2 or 3 seconds."
    ),
)

LAG23_PROMPT_TEMPLATE = LAG23_PROMPT_TEMPLATE.replace(
    "than with the three LAG profiles.",
    "than with the two LAG profiles.",
)


# ============================================================
# Use the exact existing prompt-building function
#
# Only the reference text and guidance are temporarily replaced.
# Everything else remains exactly the same.
# ============================================================

def build_lag23_prompt(case):
    global MIXED_LAG_PROMPT_TEMPLATE
    global BASE_REFERENCE_TEXT
    global GLOBAL_REFERENCE_TEXT

    original_template = MIXED_LAG_PROMPT_TEMPLATE
    original_base_text = BASE_REFERENCE_TEXT
    original_global_text = GLOBAL_REFERENCE_TEXT

    try:
        MIXED_LAG_PROMPT_TEMPLATE = (
            LAG23_PROMPT_TEMPLATE
        )

        BASE_REFERENCE_TEXT = (
            LAG23_BASE_REFERENCE_TEXT
        )

        GLOBAL_REFERENCE_TEXT = (
            LAG23_GLOBAL_REFERENCE_TEXT
        )

        return build_mixed_lag_prompt(case)

    finally:
        MIXED_LAG_PROMPT_TEMPLATE = (
            original_template
        )

        BASE_REFERENCE_TEXT = (
            original_base_text
        )

        GLOBAL_REFERENCE_TEXT = (
            original_global_text
        )


# ============================================================
# Verify the new prompt
# ============================================================

example_lag23_prompt = build_lag23_prompt(
    lag23_inference_cases[0]
)

assert "Typical NORMAL cases" in example_lag23_prompt
assert "Typical LAG +2 second cases" in example_lag23_prompt
assert "Typical LAG +3 second cases" in example_lag23_prompt

assert (
    "Typical LAG +1 second cases"
    not in example_lag23_prompt
)

assert (
    "correction near -1 second"
    not in example_lag23_prompt
)

assert (
    "approximately 1, 2, or 3 seconds"
    not in example_lag23_prompt
)

assert (
    '"label": "NORMAL or LAG"'
    in example_lag23_prompt
)

assert "gold_label" not in example_lag23_prompt
assert "conversation_id" not in example_lag23_prompt
assert "reference_profile" not in example_lag23_prompt
assert "variant" not in example_lag23_prompt

print("\nVerified:")
print("- same existing prompt-building function")
print("- only NORMAL, LAG_2 and LAG_3 references")
print("- binary NORMAL vs LAG output")
print("- no current-case label or delay in the prompt")
print("- prompt version:", LAG23_PROMPT_VERSION)

Actual LAG23 counts:
{'NORMAL': 100, 'LAG_2': 40, 'LAG_3': 40}
Cases for the new experiment:
{'NORMAL': 100, 'LAG_2': 40, 'LAG_3': 40}
Total: 180

Verified:
- same existing prompt-building function
- only NORMAL, LAG_2 and LAG_3 references
- binary NORMAL vs LAG output
- no current-case label or delay in the prompt
- prompt version: guided_independent_bc_signed_offsets_plus_global_shift_mixed_lag_v1_duration1.0_overlap0.8_threshold1.5_postmax6.0_premax2.0_shiftmin-6.0_shiftmax6.0_shiftstep0.1_scale1.5_frozen50_heldout100_NORMAL_vs_LAG2_LAG3_only_v1


In [ ]:
# ============================================================
# Run fresh inference for NORMAL vs LAG_2/LAG_3
# ============================================================

if LAG23_RESULTS_PATH.exists():
    try:
        lag23_results = json.loads(
            LAG23_RESULTS_PATH.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(
            lag23_results,
            list,
        ):
            lag23_results = []

    except Exception as exc:
        print(
            "Could not load previous LAG23 results:",
            exc,
        )

        lag23_results = []

else:
    lag23_results = []


lag23_done_case_ids = {
    result["case_id"]
    for result in lag23_results
    if (
        result.get(
            "lag23_prompt_version"
        )
        == LAG23_PROMPT_VERSION
    )
}


lag23_cases_to_run = lag23_inference_cases

if PILOT_MAX_CASES is not None:
    lag23_cases_to_run = (
        lag23_cases_to_run[
            :PILOT_MAX_CASES
        ]
    )


print(
    "Cached compatible results:",
    len(lag23_done_case_ids),
)

print(
    "Cases requested:",
    len(lag23_cases_to_run),
)

print(
    "Cases still missing:",
    sum(
        case["case_id"]
        not in lag23_done_case_ids
        for case in lag23_cases_to_run
    ),
)


for case in tqdm(
    lag23_cases_to_run,
    desc="Qwen NORMAL vs LAG +2/+3",
):
    if (
        case["case_id"]
        in lag23_done_case_ids
    ):
        continue

    prompt = build_lag23_prompt(
        case
    )

    raw_output = qwen_text_only(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    parsed = extract_json_from_text(
        raw_output
    )

    result = {
        "case_id": case["case_id"],

        "conversation_id": case[
            "conversation_id"
        ],

        "participant_A_id": case[
            "participant_A_id"
        ],

        "participant_B_id": case[
            "participant_B_id"
        ],

        # Stored only for evaluation.
        # None of these values enter the prompt.
        "variant": case["variant"],

        "reference_profile": case[
            "reference_profile"
        ],

        "gold_label": case[
            "gold_label"
        ],

        "lag_seconds": case[
            "lag_seconds"
        ],

        "lag23_prompt_version": (
            LAG23_PROMPT_VERSION
        ),

        "analysis_duration": case[
            "analysis_duration"
        ],

        "filtered_turns_A": case[
            "filtered_turns_A"
        ],

        "filtered_turns_B": case[
            "filtered_turns_B"
        ],

        "independent_bc_temporal_features": (
            case[
                "independent_bc_temporal_features"
            ]
        ),

        "global_alignment_shift_features": (
            case[
                "global_alignment_shift_features"
            ]
        ),

        "prompt": prompt,
        "raw_output": raw_output,
        "parsed": parsed,
    }

    lag23_results.append(
        result
    )

    lag23_done_case_ids.add(
        case["case_id"]
    )

    LAG23_RESULTS_PATH.write_text(
        json.dumps(
            lag23_results,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


print(
    "\nTotal stored results:",
    len(lag23_results),
)

print(
    "Saved:",
    LAG23_RESULTS_PATH,
)

Cached compatible results: 0
Cases requested: 180
Cases still missing: 180


Qwen NORMAL vs LAG +2/+3:   0%|          | 0/180 [00:00<?, ?it/s]


Total stored results: 180
Saved: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/qwen_normal_vs_lag2_lag3_binary_results.json


In [ ]:
# ============================================================
# Evaluate NORMAL vs LAG_2/LAG_3
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)


requested_lag23_case_ids = {
    case["case_id"]
    for case in lag23_cases_to_run
}


compatible_lag23_results = {}

for result in lag23_results:
    if (
        result.get(
            "lag23_prompt_version"
        )
        != LAG23_PROMPT_VERSION
    ):
        continue

    if (
        result["case_id"]
        not in requested_lag23_case_ids
    ):
        continue

    compatible_lag23_results[
        result["case_id"]
    ] = result


lag23_rows = []

for result in (
    compatible_lag23_results.values()
):
    parsed = result.get(
        "parsed",
        {},
    )

    predicted_label = (
        normalize_independent_bc_prediction(
            parsed
        )
    )

    temporal = result[
        "independent_bc_temporal_features"
    ]

    shift_features = result[
        "global_alignment_shift_features"
    ]

    lag23_rows.append({
        "case_id": result["case_id"],

        "conversation_id": result[
            "conversation_id"
        ],

        "reference_profile": result[
            "reference_profile"
        ],

        "lag_seconds": result[
            "lag_seconds"
        ],

        "gold_label": result[
            "gold_label"
        ],

        "predicted_label": (
            predicted_label
        ),

        "confidence": (
            parsed.get("confidence")
            if isinstance(parsed, dict)
            else None
        ),

        "reason": (
            parsed.get("reason")
            if isinstance(parsed, dict)
            else None
        ),

        "parse_error": (
            predicted_label is None
        ),

        "offset_list": json.dumps(
            temporal[
                "signed_strict_offsets_seconds"
            ],
            ensure_ascii=False,
        ),

        "num_offsets": temporal[
            "num_signed_strict_offsets"
        ],

        "offset_mean": temporal[
            "offset_mean_seconds"
        ],

        "offset_median": temporal[
            "offset_median_seconds"
        ],

        "offset_max": temporal[
            "offset_max_seconds"
        ],

        "offset_p75": temporal[
            "offset_p75_seconds"
        ],

        "offset_p90": temporal[
            "offset_p90_seconds"
        ],

        "percent_above_1_5": temporal[
            "percent_offsets_above_1_5_seconds"
        ],

        "clean_overlap_seconds": temporal[
            "clean_overlap_seconds"
        ],

        "clean_overlap_percent": temporal[
            "clean_overlap_percent"
        ],

        "best_B_correction_shift_seconds": (
            shift_features[
                "best_B_correction_shift_seconds"
            ]
        ),

        "estimated_B_lateness_seconds": (
            shift_features[
                "estimated_B_lateness_seconds"
            ]
        ),

        "alignment_score_gain_vs_zero": (
            shift_features[
                "alignment_score_gain_vs_zero"
            ]
        ),

        "best_num_bilateral_events": (
            shift_features[
                "best_num_bilateral_events"
            ]
        ),

        "best_event_coverage": (
            shift_features[
                "best_event_coverage"
            ]
        ),
    })


lag23_results_df = pd.DataFrame(
    lag23_rows
)

lag23_results_df.to_csv(
    LAG23_RESULTS_CSV_PATH,
    index=False,
)


print(
    "Evaluated rows:",
    len(lag23_results_df),
)

print(
    "Parse/label errors:",
    int(
        lag23_results_df[
            "parse_error"
        ].sum()
    ),
)

print(
    "Saved CSV:",
    LAG23_RESULTS_CSV_PATH,
)


valid_lag23_df = lag23_results_df[
    lag23_results_df[
        "predicted_label"
    ].isin(
        [
            "NORMAL",
            "LAG",
        ]
    )
].copy()


if len(valid_lag23_df) == 0:
    raise RuntimeError(
        "No valid NORMAL/LAG predictions were parsed."
    )


print(
    "\nOverall accuracy:",
    accuracy_score(
        valid_lag23_df[
            "gold_label"
        ],
        valid_lag23_df[
            "predicted_label"
        ],
    ),
)


print(
    "Overall balanced accuracy:",
    balanced_accuracy_score(
        valid_lag23_df[
            "gold_label"
        ],
        valid_lag23_df[
            "predicted_label"
        ],
    ),
)


print(
    "\nClassification report:"
)

print(
    classification_report(
        valid_lag23_df[
            "gold_label"
        ],

        valid_lag23_df[
            "predicted_label"
        ],

        labels=[
            "NORMAL",
            "LAG",
        ],

        zero_division=0,
    )
)


lag23_cm = confusion_matrix(
    valid_lag23_df[
        "gold_label"
    ],

    valid_lag23_df[
        "predicted_label"
    ],

    labels=[
        "NORMAL",
        "LAG",
    ],
)


lag23_confusion_df = pd.DataFrame(
    lag23_cm,

    index=[
        "Gold NORMAL",
        "Gold LAG",
    ],

    columns=[
        "Pred NORMAL",
        "Pred LAG",
    ],
)


print(
    "\nNORMAL VS LAG +2/+3 CONFUSION MATRIX"
)

display(
    lag23_confusion_df
)


lag23_profile_prediction_table = (
    pd.crosstab(
        valid_lag23_df[
            "reference_profile"
        ],

        valid_lag23_df[
            "predicted_label"
        ],
    )
    .reindex(
        LAG23_PROFILE_ORDER,
        fill_value=0,
    )
    .reindex(
        columns=[
            "NORMAL",
            "LAG",
        ],
        fill_value=0,
    )
)


lag23_profile_prediction_table.columns = [
    "Pred NORMAL",
    "Pred LAG",
]


print(
    "\nPREDICTIONS BY TRUE PROFILE"
)

display(
    lag23_profile_prediction_table
)


lag23_profile_rows = []

for profile in LAG23_PROFILE_ORDER:
    profile_df = valid_lag23_df[
        valid_lag23_df[
            "reference_profile"
        ]
        == profile
    ]

    correct_label = (
        "NORMAL"
        if profile == "NORMAL"
        else "LAG"
    )

    lag23_profile_rows.append({
        "true_profile": profile,

        "num_cases": len(
            profile_df
        ),

        "correct_predictions": int(
            (
                profile_df[
                    "predicted_label"
                ]
                == correct_label
            ).sum()
        ),

        "profile_accuracy_or_recall": (
            (
                profile_df[
                    "predicted_label"
                ]
                == correct_label
            ).mean()

            if len(profile_df)

            else None
        ),
    })


lag23_profile_metrics_df = pd.DataFrame(
    lag23_profile_rows
)


print(
    "\nPROFILE-SPECIFIC PERFORMANCE"
)

display(
    lag23_profile_metrics_df
)

Evaluated rows: 180
Parse/label errors: 0
Saved CSV: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/qwen_normal_vs_lag2_lag3_binary_results.csv

Overall accuracy: 0.8111111111111111
Overall balanced accuracy: 0.8125

Classification report:
              precision    recall  f1-score   support

      NORMAL       0.85      0.80      0.82       100
         LAG       0.77      0.82      0.80        80

    accuracy                           0.81       180
   macro avg       0.81      0.81      0.81       180
weighted avg       0.81      0.81      0.81       180


NORMAL VS LAG +2/+3 CONFUSION MATRIX


,Pred NORMAL,Pred LAG
Gold NORMAL,80,20
Gold LAG,14,66



PREDICTIONS BY TRUE PROFILE


,Pred NORMAL,Pred LAG
reference_profile,,
NORMAL,80,20
LAG_2,7,33
LAG_3,7,33



PROFILE-SPECIFIC PERFORMANCE


,true_profile,num_cases,correct_predictions,profile_accuracy_or_recall
0,NORMAL,100,80,0.800
1,LAG_2,40,33,0.825
2,LAG_3,40,33,0.825


## Transition to the Selected Reduced Temporal Representation

This notebook establishes the complete temporal development path and the richer feature packet used during exploration.

The next temporal notebook performs the **feature-reduced isolated evaluation** used in the thesis:

```text
Full development representation
(filtered turns + overlap + signed offsets + global shift)
        ↓
Remove explicit filtered turns
Remove overlap
        ↓
Selected temporal representation
(signed response-offset statistics + global alignment features)
        ↓
100 NORMAL + 100 LAG
(50 at +2 s, 50 at +3 s)
        ↓
Thesis-reported isolated temporal result
85.5% overall accuracy
```

Keeping both notebooks makes the experimental lineage explicit: this file documents **how the temporal branch was developed**, while the subsequent reduced notebook documents **the selected representation and the result reported in the thesis**.